# Experimento VQE — versão 19.5.2
## Pipeline científico: paisagem clássica → representação quântica → bancos para o Transformer

Esta versão reorganiza o projeto em uma única sequência científica. A análise clássica vem primeiro e produz hipóteses mensuráveis; somente depois o problema é codificado como QUBO/Ising, associado ao ansatz de Dicke e usado para gerar bancos de Hamiltonianos, parâmetros $\theta$ e metadados estruturais.

### Pergunta central

> Uma propriedade exata da paisagem combinatória, como o gap condicional de pares $G_{ij}$, ajuda a explicar a sensibilidade do circuito e pode gerar informação útil para treinar um Transformer que generalize entre cardinalidades e Hamiltonianos?

### Fluxo desta versão

1. reconstruir e auditar o problema financeiro original;
2. enumerar exatamente as carteiras e calcular $G_{ij}$ para os 45 pares;
3. comparar $G_{ij}$ com a antiga `decision_margin`, mantida apenas como baseline;
4. codificar o mesmo problema em QUBO/Ising e verificar equivalência;
5. construir o ansatz de Dicke e o mapa físico de cada parâmetro;
6. repetir a infraestrutura para $k\in\{2,3,4,5\}$;
7. gerar ou retomar um único banco definitivo de COBYLA por configuração;
8. exportar tabelas organizadas para análise quântica, Fourier e treinamento posterior do Transformer.


## O que foi removido e o que foi preservado

### Removido desta execução

Foram retiradas ramificações que duplicavam o mesmo experimento ou já cumpriram sua função de ablação:

- modos separados `pilot` e `robust`;
- aquecimento de subconjuntos de $\theta$ e transferência rígida de índices;
- ADAPT por comutador para Hamiltoniano diagonal;
- permutações de ordem usadas apenas para comparar AIG/DAL com AMD/COST;
- múltiplas versões do mesmo score causal e gráficos herdados dessas frentes.

Esses resultados permanecem documentados no relatório histórico, mas não são recalculados aqui.

### Preservado

Permanecem as partes que formam o pipeline científico atual:

- análise multi-$k$;
- QUBO, Hamiltoniano de Ising e solução exata;
- estado/ansatz de Dicke;
- mapa estrutural e periódico dos parâmetros;
- geração de bancos de $\theta$;
- reavaliação por `Statevector`;
- tabelas de Hamiltonianos, pares, parâmetros e ligações par–$\theta$ para o Transformer.

Há **uma única configuração experimental**. A flag de geração apenas evita iniciar centenas de otimizações sem intenção; ela não altera orçamento, amostra ou critérios científicos.


## Hipóteses pré-registradas

- **H1 — paisagem clássica:** $G_{ij}$ e as penalidades direcionais contêm mais informação que a antiga margem individual.
- **H2 — ponte clássico–quântica:** o gap do par, a posição no ansatz e o tipo de porta são fatores distintos; nenhum deles deve ser interpretado isoladamente.
- **H3 — banco multi-$k$:** variar $k$ altera o Hamiltoniano restrito, o número de parâmetros e a solução ótima; o conjunto deve ser tratado como problemas diferentes, nunca como linhas intercambiáveis.
- **H4 — restrição de $\theta_{17}$:** se a região angular associada às melhores execuções contém informação operacional, manter $\theta_{17}$ fixo nessa região enquanto o COBYLA reotimiza os outros parâmetros deve preservar ou elevar $P(x^*)$ em relação a um intervalo deslocado de controle, sem piorar sistematicamente o gap de energia.
- **H5 — Transformer:** os dados devem ser divididos por `problem_id`/Hamiltoniano, e não aleatoriamente por vetores $\theta$, para evitar vazamento.
- **H6 — utilidade:** uma descoberta só será operacionalmente relevante quando melhorar energia, probabilidade ótima ou custo de otimização contra um baseline de mesmo orçamento.


In [ ]:
# ============================================================
# 1. IMPORTS, CAMINHOS E CONFIGURAÇÃO ÚNICA
# ============================================================

from __future__ import annotations

from itertools import combinations
from pathlib import Path
from time import perf_counter
import hashlib
import json
import math
import os
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from scipy.stats import spearmanr, wilcoxon

NOTEBOOK_VERSION = "19.5.3"

# Problema original
N_ASSETS = 10
TARGET_K = 4
K_VALUES = (2, 3, 4, 5)
Q_VALUE = 0.5
RISK_FREE = 0.0475
RANDOM_SEED = 42

# Auditoria histórica do caso k=4. O bitstring está na ordem exibida pelo Qiskit.
KNOWN_EXACT_QISKIT_BITSTRING = "1001011000"
KNOWN_EXACT_ENERGY = -0.8181062577496281
STRICT_HISTORICAL_AUDIT = True

# Critérios definidos antes de olhar os rankings.
ENERGY_EQUALITY_ATOL = 1e-8
COMPETITIVE_REL_TOL = 0.01

# ÚNICO desenho para os bancos: não existe modo piloto/robusto.
BANK_RUNS_PER_CONFIGURATION = 100
COBYLA_MAXITER = 250
SHOTS = 4096
ESTIMATOR_PRECISION = float(1.0 / np.sqrt(SHOTS))
GENERATE_OR_RESUME_BANKS = False  # controle de execução, não outra versão experimental


# Experimento único de restrição de theta_17 no caso k=4.
# A flag controla somente a execução; não existem versões piloto/robusta.
RUN_THETA17_FIXED_EXPERIMENT = False
THETA17_INDEX = 17
THETA17_GRID_POINTS = 7
THETA17_RESTARTS = 20
THETA17_OPTIMAL_QUANTILE = 0.25
THETA17_INTERVAL_COVERAGE = 0.80
THETA17_MIN_OPTIMAL_ROWS = 20
THETA17_MIN_INTERVAL_FRACTION = 0.05
THETA17_DISTRIBUTION_SHOTS = SHOTS


def locate_project_file(relative_candidates, max_parent_levels=6):
    """Procura um arquivo a partir da pasta atual e das pastas superiores."""
    roots = [Path.cwd().resolve(), *list(Path.cwd().resolve().parents)[:max_parent_levels]]
    checked = []
    for candidate in relative_candidates:
        candidate = Path(candidate).expanduser()
        candidates = [candidate.resolve()] if candidate.is_absolute() else [
            (root / candidate).resolve() for root in roots
        ]
        for path in candidates:
            if path not in checked:
                checked.append(path)
            if path.is_file():
                return path, checked
    return None, checked


custom_csv = os.environ.get("VQE_STOCKS_CSV")
stock_candidates = [
    *([Path(custom_csv)] if custom_csv else []),
    Path("data/assets/stocks_price.csv"),
    Path("data/assets/stocks_prices.csv"),
    Path("../data/assets/stocks_price.csv"),
    Path("../data/assets/stocks_prices.csv"),
]
stock_path, checked_paths = locate_project_file(stock_candidates)

if stock_path is None:
    checked_text = "\n".join(f" - {path}" for path in checked_paths)
    raise FileNotFoundError(
        "O CSV de preços não foi localizado.\n"
        "Defina VQE_STOCKS_CSV ou coloque o arquivo em data/assets/.\n"
        f"Caminhos verificados:\n{checked_text}"
    )

results_root = Path(os.environ.get("VQE_RESULTS_DIR", "vqe_r"))
output_dir = results_root / "pipeline_v19_5"
classical_dir = output_dir / "01_classical"
quantum_dir = output_dir / "02_quantum"
bank_dir = output_dir / "03_banks"
transformer_dir = output_dir / "04_transformer_dataset"
theta17_dir = output_dir / "05_theta17_fixed_experiment"
figure_dir = output_dir / "figures"
for directory in [output_dir, classical_dir, quantum_dir, bank_dir, transformer_dir, theta17_dir, figure_dir]:
    directory.mkdir(parents=True, exist_ok=True)

CONFIG = {
    "notebook_version": NOTEBOOK_VERSION,
    "n_assets": N_ASSETS,
    "target_k": TARGET_K,
    "k_values": list(K_VALUES),
    "q_value": Q_VALUE,
    "risk_free": RISK_FREE,
    "random_seed": RANDOM_SEED,
    "strict_historical_audit": STRICT_HISTORICAL_AUDIT,
    "energy_equality_atol": ENERGY_EQUALITY_ATOL,
    "competitive_relative_tolerance": COMPETITIVE_REL_TOL,
    "bank_runs_per_configuration": BANK_RUNS_PER_CONFIGURATION,
    "cobyla_maxiter": COBYLA_MAXITER,
    "shots": SHOTS,
    "estimator_precision": ESTIMATOR_PRECISION,
    "generate_or_resume_banks": GENERATE_OR_RESUME_BANKS,
    "run_theta17_fixed_experiment": RUN_THETA17_FIXED_EXPERIMENT,
    "theta17_index": THETA17_INDEX,
    "theta17_grid_points": THETA17_GRID_POINTS,
    "theta17_restarts": THETA17_RESTARTS,
    "theta17_optimal_quantile": THETA17_OPTIMAL_QUANTILE,
    "theta17_interval_coverage": THETA17_INTERVAL_COVERAGE,
    "theta17_min_optimal_rows": THETA17_MIN_OPTIMAL_ROWS,
    "theta17_min_interval_fraction": THETA17_MIN_INTERVAL_FRACTION,
    "theta17_distribution_shots": THETA17_DISTRIBUTION_SHOTS,
}

print(json.dumps(CONFIG, indent=2, ensure_ascii=False))
print("CSV:", stock_path.resolve())
print("saídas:", output_dir.resolve())


# Parte I — reconstrução clássica exata

Esta parte não usa circuito quântico. Ela define a paisagem combinatória que qualquer método quântico ou Transformer deverá reproduzir.

A função objetivo preserva exatamente as escolhas do experimento original:

\[
E(x)=q\,x^T\Sigma x-(1-q)\,\mu^Tx+r_f,
\qquad \sum_i x_i=k.
\]

O retorno usado é a **soma dos retornos diários**, não a média. Alterar essa escolha cria outro Hamiltoniano.


In [ ]:
# ============================================================
# 2. PREÇOS, RETORNOS E COVARIÂNCIA
# ============================================================

stocks_prices = pd.read_csv(stock_path, index_col=0)
stocks_prices = (
    stocks_prices
    .apply(pd.to_numeric, errors="coerce")
    .replace([np.inf, -np.inf], np.nan)
    .dropna(axis=1, how="all")
    .ffill()
    .bfill()
    .dropna()
)

if stocks_prices.empty:
    raise ValueError("O CSV não contém dados numéricos válidos.")
if stocks_prices.columns.duplicated().any():
    raise ValueError("O CSV possui tickers repetidos.")
if stocks_prices.shape[1] != N_ASSETS:
    raise ValueError(
        f"Esperados {N_ASSETS} ativos; encontrados {stocks_prices.shape[1]}: "
        f"{stocks_prices.columns.tolist()}"
    )
if len(stocks_prices) < 3:
    raise ValueError("São necessárias ao menos três datas.")

daily_returns = stocks_prices.pct_change(fill_method=None).dropna()
assets_returns = daily_returns.sum()
covariance = daily_returns.cov()

mu = assets_returns.to_numpy(dtype=float)
sigma = covariance.to_numpy(dtype=float)
sigma = 0.5 * (sigma + sigma.T)
tickers = list(stocks_prices.columns)

if not np.all(np.isfinite(mu)) or not np.all(np.isfinite(sigma)):
    raise ValueError("Retornos ou covariância contêm valores não finitos.")

min_cov_eigenvalue = float(np.linalg.eigvalsh(sigma).min())
if min_cov_eigenvalue < -1e-10:
    raise ValueError(
        "A covariância não é semidefinida positiva dentro da tolerância: "
        f"lambda_min={min_cov_eigenvalue:.3e}"
    )

asset_summary_df = pd.DataFrame({
    "asset_index": np.arange(N_ASSETS, dtype=int),
    "ticker": tickers,
    "return_sum": mu,
    "variance": np.diag(sigma),
    "risk_connection_abs": np.sum(np.abs(sigma), axis=1),
})

DATA_HASH = hashlib.sha256(
    np.concatenate([mu, sigma.reshape(-1)]).astype(np.float64).tobytes()
).hexdigest()[:16]

print("preços:", stocks_prices.shape)
print("retornos diários:", daily_returns.shape)
print("tickers:", tickers)
print("data_hash:", DATA_HASH)
print("menor autovalor da covariância:", f"{min_cov_eigenvalue:.3e}")
display(asset_summary_df)


## 3. Funções exatas compartilhadas por todos os valores de $k$

A mesma infraestrutura é usada no caso original $k=4$ e, posteriormente, nos contextos multi-$k$. Isso evita manter versões diferentes da função objetivo ou do cálculo de $G_{ij}$.

Para cada par $(i,j)$ são calculados os quatro mínimos condicionados:

\[
E_{ij}^{ab}=\min_{x_i=a,x_j=b,\,\sum x=k}E(x),\qquad ab\in\{00,01,10,11\}.
\]

O gap principal é

\[
G_{ij}=E_{ij}^{\text{segundo estado}}-E^*.
\]

Também são preservadas as quatro penalidades direcionais, a inversão conjunta, a não aditividade e o número de trocas necessárias para sair do ótimo.


In [ ]:
# ============================================================
# 3. ENUMERAÇÃO, MARGEM ANTIGA E GAPS CONDICIONAIS
# ============================================================

PAIR_STATES = ("00", "01", "10", "11")


def portfolio_objective(x_binary, mu_values=mu, sigma_values=sigma):
    x = np.asarray(x_binary, dtype=float).reshape(-1)
    return float(
        Q_VALUE * x @ sigma_values @ x
        - (1.0 - Q_VALUE) * x @ mu_values
        + RISK_FREE
    )


def bitstring_from_x(x):
    return "".join(str(int(value)) for value in np.asarray(x, dtype=int))


def selected_assets_from_x(x, labels=tickers):
    return [ticker for ticker, selected in zip(labels, x) if int(selected) == 1]


def enumerate_portfolios(k_value, mu_values=mu, sigma_values=sigma, labels=tickers):
    n_value = len(labels)
    rows = []
    for selected_indices in combinations(range(n_value), int(k_value)):
        x = np.zeros(n_value, dtype=int)
        x[list(selected_indices)] = 1
        bits = bitstring_from_x(x)
        rows.append({
            "selected_indices": tuple(int(i) for i in selected_indices),
            "x_asset_order": tuple(int(v) for v in x),
            "bitstring_asset_order": bits,
            "bitstring_qiskit_order": bits[::-1],
            "selected_assets": tuple(selected_assets_from_x(x, labels)),
            "objective": portfolio_objective(x, mu_values, sigma_values),
        })
    return (
        pd.DataFrame(rows)
        .sort_values(["objective", "bitstring_asset_order"])
        .reset_index(drop=True)
    )


def old_asset_decision_margins(exact_x, exact_energy, energy_span, labels=tickers):
    selected_indices = np.flatnonzero(exact_x == 1)
    excluded_indices = np.flatnonzero(exact_x == 0)
    rows = []
    for asset_index in range(len(labels)):
        alternatives = []
        if exact_x[asset_index] == 1:
            for replacement in excluded_indices:
                trial = exact_x.copy()
                trial[asset_index] = 0
                trial[replacement] = 1
                alternatives.append((portfolio_objective(trial), trial))
        else:
            for removed in selected_indices:
                trial = exact_x.copy()
                trial[asset_index] = 1
                trial[removed] = 0
                alternatives.append((portfolio_objective(trial), trial))
        best_energy, best_trial = min(alternatives, key=lambda item: item[0])
        margin = max(float(best_energy - exact_energy), 0.0)
        rows.append({
            "asset_index": int(asset_index),
            "ticker": labels[asset_index],
            "selected_exact": int(exact_x[asset_index]),
            "decision_margin": margin,
            "decision_margin_relative": margin / max(energy_span, ENERGY_EQUALITY_ATOL),
            "best_flip_bitstring": bitstring_from_x(best_trial),
            "best_flip_selected_assets": tuple(selected_assets_from_x(best_trial, labels)),
            "return_coefficient": float(-(1.0 - Q_VALUE) * mu[asset_index]),
            "self_risk_coefficient": float(Q_VALUE * sigma[asset_index, asset_index]),
            "risk_connection": float(Q_VALUE * np.sum(np.abs(sigma[asset_index]))),
        })
    return pd.DataFrame(rows).sort_values("decision_margin", ascending=False).reset_index(drop=True)


def build_pair_gap_table(enumeration_df, exact_x, exact_energy, energy_span, asset_margin_df, labels=tickers):
    n_value = len(labels)
    margin_lookup = asset_margin_df.set_index("asset_index")["decision_margin"].to_dict()

    def conditional_best(i, j, state):
        a, b = int(state[0]), int(state[1])
        mask = enumeration_df["x_asset_order"].map(
            lambda x: int(x[i]) == a and int(x[j]) == b
        )
        subset = enumeration_df.loc[mask]
        if subset.empty:
            raise RuntimeError(f"Estado inviável para par {(i, j)}: {state}")
        best = subset.iloc[0]
        return {
            "count": int(len(subset)),
            "energy": float(best["objective"]),
            "bitstring": str(best["bitstring_asset_order"]),
            "selected_assets": tuple(best["selected_assets"]),
        }

    rows = []
    for i, j in combinations(range(n_value), 2):
        state_results = {state: conditional_best(i, j, state) for state in PAIR_STATES}
        ordered_states = sorted(PAIR_STATES, key=lambda s: (state_results[s]["energy"], s))
        best_state, second_state = ordered_states[:2]
        best_energy = state_results[best_state]["energy"]
        second_energy = state_results[second_state]["energy"]
        gij = max(float(second_energy - best_energy), 0.0)

        states_at_best = tuple(
            state for state in PAIR_STATES
            if np.isclose(state_results[state]["energy"], best_energy, atol=ENERGY_EQUALITY_ATOL, rtol=0.0)
        )
        second_x = np.asarray([int(v) for v in state_results[second_state]["bitstring"]], dtype=int)
        hamming = int(np.sum(second_x != exact_x))
        removed = [labels[idx] for idx in np.flatnonzero((exact_x == 1) & (second_x == 0))]
        added = [labels[idx] for idx in np.flatnonzero((exact_x == 0) & (second_x == 1))]
        n_swaps = hamming // 2

        global_pair_state = f"{exact_x[i]}{exact_x[j]}"
        decision_pattern = {
            "00": "exclude_exclude",
            "01": "exclude_include",
            "10": "include_exclude",
            "11": "include_include",
        }[global_pair_state]
        a_star, b_star = int(global_pair_state[0]), int(global_pair_state[1])
        single_i_state = f"{1-a_star}{b_star}"
        single_j_state = f"{a_star}{1-b_star}"
        joint_state = f"{1-a_star}{1-b_star}"
        single_i_gap = max(state_results[single_i_state]["energy"] - exact_energy, 0.0)
        single_j_gap = max(state_results[single_j_state]["energy"] - exact_energy, 0.0)
        joint_gap = max(state_results[joint_state]["energy"] - exact_energy, 0.0)
        nonadditivity = joint_gap - single_i_gap - single_j_gap
        exit_gap = min(single_i_gap, single_j_gap, joint_gap)
        delta_01_10 = abs(state_results["01"]["energy"] - state_results["10"]["energy"])
        top2_swap = set(ordered_states[:2]) == {"01", "10"}

        rows.append({
            "asset_i": labels[i], "asset_j": labels[j],
            "asset_index_i": int(i), "asset_index_j": int(j),
            "pair": f"{labels[i]}/{labels[j]}",
            "global_pair_state": global_pair_state,
            "decision_pattern": decision_pattern,
            "single_flip_i_state": single_i_state,
            "single_flip_j_state": single_j_state,
            "joint_complement_state": joint_state,
            "single_flip_i_gap": float(single_i_gap),
            "single_flip_j_gap": float(single_j_gap),
            "joint_flip_gap": float(joint_gap),
            "joint_flip_gap_relative": float(joint_gap / max(energy_span, ENERGY_EQUALITY_ATOL)),
            "joint_flip_nonadditivity": float(nonadditivity),
            "joint_flip_nonadditivity_relative": float(nonadditivity / max(energy_span, ENERGY_EQUALITY_ATOL)),
            "G_ij_decomposition_error": float(abs(gij - exit_gap)),
            **{f"E_{state}": state_results[state]["energy"] for state in PAIR_STATES},
            **{f"delta_{state}": state_results[state]["energy"] - exact_energy for state in PAIR_STATES},
            **{f"count_{state}": state_results[state]["count"] for state in PAIR_STATES},
            **{f"bitstring_{state}": state_results[state]["bitstring"] for state in PAIR_STATES},
            "best_pair_state": best_state,
            "second_best_pair_state": second_state,
            "states_at_best": states_at_best,
            "G_ij": gij,
            "G_ij_relative": gij / max(energy_span, ENERGY_EQUALITY_ATOL),
            "swap_state_energy_difference": delta_01_10,
            "swap_state_difference_relative": delta_01_10 / max(energy_span, ENERGY_EQUALITY_ATOL),
            "top2_are_01_10": bool(top2_swap),
            "second_best_conditional_bitstring": state_results[second_state]["bitstring"],
            "second_best_selected_assets": state_results[second_state]["selected_assets"],
            "hamming_distance_from_global_optimum": hamming,
            "number_of_asset_swaps": int(n_swaps),
            "assets_removed": tuple(removed),
            "assets_added": tuple(added),
            "G_ij_per_swap": gij / max(n_swaps, 1),
            "old_margin_i": float(margin_lookup[i]),
            "old_margin_j": float(margin_lookup[j]),
            "old_pair_margin_min": float(min(margin_lookup[i], margin_lookup[j])),
            "old_pair_margin_sum": float(margin_lookup[i] + margin_lookup[j]),
            "opposite_decisions": bool(exact_x[i] != exact_x[j]),
        })

    table = pd.DataFrame(rows)
    table["is_degenerate"] = table["states_at_best"].map(len).gt(1)
    table["is_ambiguous"] = (~table["is_degenerate"] & table["G_ij_relative"].le(COMPETITIVE_REL_TOL))
    table["is_competitive_swap"] = (
        table["top2_are_01_10"] & table["swap_state_difference_relative"].le(COMPETITIVE_REL_TOL)
    )

    def classify(row):
        if row["is_degenerate"]:
            return "degenerate"
        if row["is_ambiguous"]:
            return "ambiguous"
        if row["is_competitive_swap"]:
            return "competitive_swap"
        return row["decision_pattern"]

    table["pair_class"] = table.apply(classify, axis=1)
    table = table.sort_values(["G_ij", "pair"], ascending=[False, True]).reset_index(drop=True)
    table["G_ij_rank"] = np.arange(1, len(table) + 1, dtype=int)
    return table


def build_classical_analysis(k_value):
    enumeration = enumerate_portfolios(k_value)
    exact_energy = float(enumeration.loc[0, "objective"])
    optimal_mask = np.isclose(
        enumeration["objective"].to_numpy(dtype=float), exact_energy,
        atol=ENERGY_EQUALITY_ATOL, rtol=0.0,
    )
    optimal = enumeration.loc[optimal_mask].copy()
    exact_asset_bitstrings = sorted(optimal["bitstring_asset_order"].astype(str).unique())
    exact_qiskit_bitstrings = sorted(optimal["bitstring_qiskit_order"].astype(str).unique())
    exact_x = np.asarray([int(v) for v in exact_asset_bitstrings[0]], dtype=int)
    energy_span = float(enumeration["objective"].max() - exact_energy)
    margins = old_asset_decision_margins(exact_x, exact_energy, energy_span)
    pairs = build_pair_gap_table(enumeration, exact_x, exact_energy, energy_span, margins)
    return {
        "k": int(k_value),
        "enumeration_df": enumeration,
        "optimal_df": optimal,
        "exact_energy": exact_energy,
        "energy_span": energy_span,
        "exact_x": exact_x,
        "exact_asset_bitstrings": exact_asset_bitstrings,
        "exact_qiskit_bitstrings": exact_qiskit_bitstrings,
        "selected_assets": selected_assets_from_x(exact_x),
        "asset_decision_df": margins,
        "pair_table": pairs,
    }


## 4. Caso original $k=4$: auditoria histórica e análise dos 45 pares

O caso $k=4$ é executado primeiro porque possui energia e bitstring históricos conhecidos. A auditoria é dividida em duas camadas independentes:

1. **Auditoria estrutural obrigatória:** contagem das 210 carteiras, 45 pares, estados condicionados, não negatividade dos gaps, paridade da distância de Hamming e decomposição de $G_{ij}$. Essa camada sempre precisa passar.
2. **Auditoria histórica opcional:** compara energia e bitstring com o experimento original. Ela só bloqueia a execução quando `STRICT_HISTORICAL_AUDIT=True`.

Com `STRICT_HISTORICAL_AUDIT=False`, o notebook pode ser executado em outras janelas temporais, matrizes de covariância e Hamiltonianos sem editar manualmente os valores históricos. As divergências históricas continuam registradas no relatório de auditoria, mas não são tratadas como falha estrutural.


In [ ]:
# ============================================================
# 4. EXECUTAR E AUDITAR O CASO ORIGINAL k=4
# ============================================================

target_classical = build_classical_analysis(TARGET_K)
enumeration_df = target_classical["enumeration_df"]
asset_decision_df = target_classical["asset_decision_df"]
pair_table = target_classical["pair_table"]
EXACT_ENERGY = target_classical["exact_energy"]
ENERGY_SPAN = target_classical["energy_span"]
EXACT_X = target_classical["exact_x"]
EXACT_ASSET_BITSTRINGS = target_classical["exact_asset_bitstrings"]
EXACT_QISKIT_BITSTRINGS = target_classical["exact_qiskit_bitstrings"]
EXACT_SELECTED_ASSETS = target_classical["selected_assets"]

energy_matches_history = np.isclose(EXACT_ENERGY, KNOWN_EXACT_ENERGY, atol=1e-10, rtol=0.0)
bitstring_matches_history = KNOWN_EXACT_QISKIT_BITSTRING in EXACT_QISKIT_BITSTRINGS
expected_counts = {
    "00": math.comb(N_ASSETS - 2, TARGET_K),
    "01": math.comb(N_ASSETS - 2, TARGET_K - 1),
    "10": math.comb(N_ASSETS - 2, TARGET_K - 1),
    "11": math.comb(N_ASSETS - 2, TARGET_K - 2),
}
energy_columns = [f"E_{state}" for state in PAIR_STATES]
delta_columns = [f"delta_{state}" for state in PAIR_STATES]

historical_audit = {
    "historical_energy_match_ok": bool(energy_matches_history),
    "historical_bitstring_match_ok": bool(bitstring_matches_history),
}

structural_audit = {
    "portfolio_count_ok": len(enumeration_df) == math.comb(N_ASSETS, TARGET_K),
    "pair_count_ok": len(pair_table) == math.comb(N_ASSETS, 2),
    "state_counts_ok": all((pair_table[f"count_{state}"] == count).all() for state, count in expected_counts.items()),
    "conditional_minimum_is_global_ok": bool(np.allclose(
        pair_table[energy_columns].min(axis=1), EXACT_ENERGY,
        atol=ENERGY_EQUALITY_ATOL, rtol=0.0,
    )),
    "nonnegative_deltas_ok": bool((pair_table[delta_columns].min(axis=1) >= -ENERGY_EQUALITY_ATOL).all()),
    "even_hamming_distance_ok": bool((pair_table["hamming_distance_from_global_optimum"] % 2 == 0).all()),
    "G_ij_decomposition_ok": bool(pair_table["G_ij_decomposition_error"].le(ENERGY_EQUALITY_ATOL).all()),
}

audit = {
    "strict_historical_audit": bool(STRICT_HISTORICAL_AUDIT),
    **historical_audit,
    **structural_audit,
}

failed_structural = [
    name
    for name, value in structural_audit.items()
    if name.endswith("_ok") and not value
]
if failed_structural:
    raise RuntimeError(
        f"Auditoria estrutural clássica falhou: {failed_structural}"
    )

failed_historical = [
    name
    for name, value in historical_audit.items()
    if name.endswith("_ok") and not value
]
if STRICT_HISTORICAL_AUDIT and failed_historical:
    raise RuntimeError(
        "Auditoria histórica clássica falhou: "
        f"{failed_historical}. Revise CSV, ordem das colunas, "
        "retorno=sum(), q e risk_free, ou use "
        "STRICT_HISTORICAL_AUDIT=False para analisar outro Hamiltoniano."
    )

if failed_historical and not STRICT_HISTORICAL_AUDIT:
    warnings.warn(
        "Os valores atuais não coincidem com o caso histórico original, "
        "mas a execução continuará porque STRICT_HISTORICAL_AUDIT=False. "
        f"Divergências: {failed_historical}",
        RuntimeWarning,
    )

print("energia exata:", EXACT_ENERGY)
print("bitstring(s) Qiskit:", EXACT_QISKIT_BITSTRINGS)
print("ativos selecionados:", EXACT_SELECTED_ASSETS)
print(json.dumps(audit, indent=2, ensure_ascii=False))
display(enumeration_df.head(10))
display(asset_decision_df)
display(pair_table.head(15))


## 5. Baseline antigo, categorias e pares de referência

A antiga `decision_margin` permanece apenas para ablação. Ela não seleciona mais sozinha os pares do experimento.

A seleção objetiva inclui:

- maior $G_{ij}$;
- maior custo de inversão conjunta;
- maior não aditividade;
- exemplos inclusão–exclusão, inclusão–inclusão e exclusão–exclusão;
- par competitivo e par ambíguo;
- AMD/COST, AIG/DAL e COST/DAL apenas como referências históricas.


In [ ]:
# ============================================================
# 5. RANKING, ABLAÇÃO E VISUALIZAÇÃO CLÁSSICA
# ============================================================


def first_or_none(df, sort_column, ascending=False):
    if df.empty:
        return None
    return df.sort_values(sort_column, ascending=ascending).iloc[0]


def selection_record(role, row):
    if row is None:
        return {"selection_role": role, "pair": None, "status": "not_found"}
    return {
        "selection_role": role,
        "pair": row["pair"],
        "asset_i": row["asset_i"],
        "asset_j": row["asset_j"],
        "asset_index_i": int(row["asset_index_i"]),
        "asset_index_j": int(row["asset_index_j"]),
        "decision_pattern": row["decision_pattern"],
        "pair_class": row["pair_class"],
        "G_ij": float(row["G_ij"]),
        "G_ij_relative": float(row["G_ij_relative"]),
        "joint_flip_gap": float(row["joint_flip_gap"]),
        "joint_flip_nonadditivity": float(row["joint_flip_nonadditivity"]),
        "old_pair_margin_min": float(row["old_pair_margin_min"]),
        "number_of_asset_swaps": int(row["number_of_asset_swaps"]),
        "status": "selected",
    }

nondegenerate = pair_table.loc[~pair_table["is_degenerate"]]
selections = [
    selection_record("largest_G_pair", first_or_none(nondegenerate, "G_ij")),
    selection_record("largest_joint_flip_gap", first_or_none(nondegenerate, "joint_flip_gap")),
    selection_record(
        "largest_abs_joint_nonadditivity",
        None if nondegenerate.empty else nondegenerate.loc[nondegenerate["joint_flip_nonadditivity"].abs().idxmax()],
    ),
    selection_record(
        "strong_exclude_include_pair",
        first_or_none(nondegenerate.loc[nondegenerate["decision_pattern"].isin(["exclude_include", "include_exclude"])], "G_ij"),
    ),
    selection_record("strong_include_include_pair", first_or_none(nondegenerate.loc[nondegenerate["decision_pattern"].eq("include_include")], "G_ij")),
    selection_record("strong_exclude_exclude_pair", first_or_none(nondegenerate.loc[nondegenerate["decision_pattern"].eq("exclude_exclude")], "G_ij")),
    selection_record("competitive_pair", first_or_none(pair_table.loc[pair_table["is_competitive_swap"]], "swap_state_difference_relative", ascending=True)),
    selection_record("most_ambiguous_pair", first_or_none(nondegenerate, "G_ij_relative", ascending=True)),
]
for role, names in {
    "reference_AMD_COST": {"AMD", "COST"},
    "reference_AIG_DAL": {"AIG", "DAL"},
    "reference_COST_DAL": {"COST", "DAL"},
}.items():
    match = pair_table.loc[pair_table.apply(lambda row: {row["asset_i"], row["asset_j"]} == names, axis=1)]
    selections.append(selection_record(role, None if match.empty else match.iloc[0]))

pair_category_selection_df = pd.DataFrame(selections)
rho_min, p_min = spearmanr(pair_table["G_ij"], pair_table["old_pair_margin_min"])
rho_sum, p_sum = spearmanr(pair_table["G_ij"], pair_table["old_pair_margin_sum"])
ablation_summary_df = pd.DataFrame([
    {"baseline": "old_pair_margin_min", "spearman_rho_with_G_ij": rho_min, "p_value": p_min},
    {"baseline": "old_pair_margin_sum", "spearman_rho_with_G_ij": rho_sum, "p_value": p_sum},
])

top_pairs = pair_table.nlargest(15, "G_ij").sort_values("G_ij")
fig, ax = plt.subplots(figsize=(9, 6))
ax.barh(top_pairs["pair"], top_pairs["G_ij"])
ax.set_xlabel(r"$G_{ij}$")
ax.set_ylabel("par de ativos")
ax.set_title("15 maiores gaps condicionais — k=4")
fig.tight_layout()
fig.savefig(figure_dir / "01_ranking_Gij_k4.png", dpi=180, bbox_inches="tight")
plt.show()

gap_matrix = np.full((N_ASSETS, N_ASSETS), np.nan, dtype=float)
for _, row in pair_table.iterrows():
    i, j = int(row["asset_index_i"]), int(row["asset_index_j"])
    gap_matrix[i, j] = gap_matrix[j, i] = float(row["G_ij"])
np.fill_diagonal(gap_matrix, 0.0)
fig, ax = plt.subplots(figsize=(8, 7))
image = ax.imshow(gap_matrix)
ax.set_xticks(range(N_ASSETS), tickers, rotation=45, ha="right")
ax.set_yticks(range(N_ASSETS), tickers)
ax.set_title(r"Mapa dos gaps condicionais $G_{ij}$ — k=4")
fig.colorbar(image, ax=ax, label=r"$G_{ij}$")
fig.tight_layout()
fig.savefig(figure_dir / "02_heatmap_Gij_k4.png", dpi=180, bbox_inches="tight")
plt.show()

display(pair_category_selection_df)
display(ablation_summary_df)


## Decisão científica após a parte clássica

O ranking clássico **não é interpretado como ranking de portas**. Ele apenas define pares e categorias para testes quânticos posteriores.

A ponte que ainda precisa ser testada é:

\[
G_{ij}\;\overset{?}{\longrightarrow}\;
\text{sensibilidade de }\langle H\rangle\text{ e }P(x^*)
\;\overset{?}{\longrightarrow}\;
\text{melhoria real do VQE}.
\]

A partir daqui, cada resultado deve manter juntos:

- identidade do Hamiltoniano;
- cardinalidade $k$;
- estrutura física do ansatz;
- vetor $\theta$;
- energia e probabilidade avaliadas exatamente.


# Parte II — codificação QUBO/Ising e auditoria quântica

Esta seção importa Qiskit e DOcplex somente depois de a análise clássica ter passado. O Hamiltoniano deve reproduzir exatamente a enumeração e conter apenas termos diagonais $I$, $Z$ e $ZZ$.


In [ ]:
# ============================================================
# 6. DEPENDÊNCIAS QUÂNTICAS
# ============================================================

from docplex.mp.model import Model
from qiskit import QuantumCircuit, QuantumRegister
from qiskit.circuit import ParameterVector
from qiskit.primitives import BackendEstimatorV2
from qiskit.quantum_info import Statevector
from qiskit_aer import AerSimulator
from qiskit_optimization.translators import from_docplex_mp
from qiskit_optimization.converters import QuadraticProgramToQubo

try:
    from qiskit_algorithms.minimum_eigensolvers import NumPyMinimumEigensolver
    from qiskit_algorithms.optimizers import COBYLA
except ImportError:
    from qiskit.algorithms.minimum_eigensolvers import NumPyMinimumEigensolver
    from qiskit.algorithms.optimizers import COBYLA

try:
    import cvxpy as cp
except ImportError:
    cp = None
    warnings.warn("CVXPY não está instalado; a enumeração continuará sendo a referência clássica.")


In [ ]:
# ============================================================
# 7. CONSTRUIR QUBO/ISING PARA UMA CARDINALIDADE
# ============================================================


def build_docplex_ising(k_value):
    model = Model(name=f"portfolio_n{N_ASSETS}_k{k_value}")
    variables = np.array([
        model.binary_var(name=f"x_{index}") for index in range(N_ASSETS)
    ], dtype=object)

    risk_expression = model.sum(
        float(sigma[row, column]) * variables[row] * variables[column]
        for row in range(N_ASSETS) for column in range(N_ASSETS)
    )
    return_expression = model.sum(
        float(mu[index]) * variables[index] for index in range(N_ASSETS)
    )
    model.minimize(
        Q_VALUE * risk_expression
        - (1.0 - Q_VALUE) * return_expression
        + RISK_FREE
    )
    model.add_constraint(model.sum(variables.tolist()) == int(k_value), ctname="budget")

    quadratic_program = from_docplex_mp(model=model)
    qubo = QuadraticProgramToQubo().convert(quadratic_program)
    ising, offset = qubo.to_ising()

    labels = [str(label) for label in ising.paulis.to_labels()]
    non_diagonal = [label for label in labels if "X" in label or "Y" in label]
    if non_diagonal:
        raise RuntimeError(f"Hamiltoniano não diagonal: {non_diagonal[:10]}")

    exact_result = NumPyMinimumEigensolver().compute_minimum_eigenvalue(operator=ising)
    exact_energy = float(np.real(exact_result.eigenvalue + offset))
    state_dict = exact_result.eigenstate.to_dict()
    exact_bitstrings = sorted({
        str(bitstring).replace(" ", "")
        for bitstring, amplitude in state_dict.items()
        if abs(amplitude) > 1e-12
    })

    return {
        "model": model,
        "quadratic_program": quadratic_program,
        "qubo": qubo,
        "ising": ising,
        "offset": float(offset),
        "pauli_labels": labels,
        "exact_energy_ising": exact_energy,
        "exact_bitstrings_ising": exact_bitstrings,
    }


target_encoding = build_docplex_ising(TARGET_K)
if not np.isclose(target_encoding["exact_energy_ising"], EXACT_ENERGY, atol=1e-10, rtol=0.0):
    raise RuntimeError("A energia Ising não coincide com a enumeração clássica.")
if not set(EXACT_QISKIT_BITSTRINGS).intersection(target_encoding["exact_bitstrings_ising"]):
    raise RuntimeError("O bitstring Ising não coincide com a enumeração clássica.")

lp_path = quantum_dir / "portfolio_k4.lp"
lp_path.write_text(target_encoding["model"].export_as_lp_string(), encoding="utf-8")

print("energia clássica:", EXACT_ENERGY)
print("energia Ising + offset:", target_encoding["exact_energy_ising"])
print("bitstrings Ising:", target_encoding["exact_bitstrings_ising"])
print("termos de Pauli:", len(target_encoding["pauli_labels"]))
print("LP salvo em:", lp_path.resolve())


# Parte III — ansatz de Dicke e mapa físico dos parâmetros

O índice `theta_index` não é tratado como propriedade física. Cada parâmetro é descrito por:

- bloco original do ansatz (`CY` ou `CCY`);
- qubits e distância do bloco;
- instruções primitivas após decomposição;
- número de ocorrências;
- período angular máximo/garantido.

Nesta implementação:

- `CY` contém uma `CRY(θ)`: período máximo $4\pi$, classe `four_pi_eligible`;
- `CCY` contém `RY(θ)`, `CCX`, `RY(-θ)`, `CCX`: equivale a `CCRY(2θ)`, possui período $2\pi$ garantido e duas ocorrências primitivas.


In [ ]:
# ============================================================
# 8. PORTAS E ANSATZ DE DICKE RASTREÁVEL
# ============================================================


def CY_parameterized(identifier):
    param = ParameterVector(name=f"x{identifier}", length=1)
    qc = QuantumCircuit(2)
    qc.cry(param[0], 1, 0)
    return qc.to_gate(label="CY")


def CCY_parameterized(identifier):
    param = ParameterVector(name=f"y{identifier}", length=1)
    qc = QuantumCircuit(3)
    qc.ry(param[0], 0)
    qc.ccx(2, 1, 0)
    qc.ry(-param[0], 0)
    qc.ccx(2, 1, 0)
    return qc.to_gate(label="CCY")


def dicke_parameter_count(n_value, k_value):
    return int(k_value * (2 * n_value - k_value - 1) / 2)


def build_tracked_dicke_ansatz(n_value, k_value, seed):
    numpy_state = np.random.get_state()
    try:
        np.random.seed(int(seed))
        qr = QuantumRegister(n_value, "q")
        qc = QuantumCircuit(qr)
        for excitation_index in range(k_value):
            qc.x(n_value - excitation_index - 1)

        records = []
        aux = 1
        for l_value in range(n_value)[::-1]:
            for i_value in range(l_value - 1, l_value - 1 - k_value, -1):
                if i_value >= 0:
                    unique_name = f"{l_value}{i_value}{aux}{np.random.randint(0, int(1e8))}"
                    if i_value == l_value - 1:
                        gate = CY_parameterized(unique_name)
                        gate_parameter = list(gate.params[0].parameters)[0]
                        qc.cx(qr[i_value], qr[l_value])
                        qc.append(gate, [qr[i_value], qr[l_value]])
                        qc.cx(qr[i_value], qr[l_value])
                        gate_type = "CY"
                    else:
                        gate = CCY_parameterized(unique_name)
                        gate_parameter = list(gate.params[0].parameters)[0]
                        qc.cx(qr[i_value], qr[l_value])
                        qc.append(gate, [qr[i_value], qr[i_value + 1], qr[l_value]])
                        qc.cx(qr[i_value], qr[l_value])
                        gate_type = "CCY"
                    records.append({
                        "parameter_object": gate_parameter,
                        "parameter_name": str(gate_parameter),
                        "l": int(l_value),
                        "i": int(i_value),
                        "distance": int(l_value - i_value),
                        "ansatz_gate_type": gate_type,
                    })
                aux += 1
    finally:
        np.random.set_state(numpy_state)

    ordered_parameters = list(qc.parameters)
    parameter_to_index = {parameter: index for index, parameter in enumerate(ordered_parameters)}
    structure_rows = []
    for record in records:
        parameter_object = record.pop("parameter_object")
        structure_rows.append({
            "theta_index": int(parameter_to_index[parameter_object]),
            **record,
        })
    structure_df = pd.DataFrame(structure_rows).sort_values("theta_index").reset_index(drop=True)
    expected = dicke_parameter_count(n_value, k_value)
    if len(structure_df) != expected:
        raise RuntimeError(f"Esperados {expected} parâmetros; encontrados {len(structure_df)}.")
    structure_df["n"] = int(n_value)
    structure_df["k"] = int(k_value)
    structure_df["rho_k_over_n"] = k_value / n_value
    structure_df["u_l"] = structure_df["l"] / max(n_value - 1, 1)
    structure_df["u_distance"] = structure_df["distance"] / max(k_value, 1)
    structure_df["u_theta_index"] = structure_df["theta_index"] / max(expected - 1, 1)
    return qc.decompose(), structure_df


In [ ]:
# ============================================================
# 9. MAPA FÍSICO PÓS-DECOMPOSIÇÃO E AUDITORIA DE PERIODICIDADE
# ============================================================


def build_physical_parameter_map(ansatz, structure_df):
    parameter_order = list(ansatz.parameters)
    parameter_to_index = {parameter: index for index, parameter in enumerate(parameter_order)}
    occurrence_rows = []

    for instruction_position, instruction in enumerate(ansatz.data):
        operation = instruction.operation
        operation_name = str(operation.name).lower()
        qubits = tuple(int(ansatz.find_bit(qubit).index) for qubit in instruction.qubits)
        for parameter_slot, expression in enumerate(operation.params):
            expression_parameters = getattr(expression, "parameters", set())
            for parameter in expression_parameters:
                if parameter not in parameter_to_index:
                    continue
                try:
                    coefficient = float(expression.gradient(parameter))
                except Exception:
                    coefficient = np.nan
                occurrence_rows.append({
                    "theta_index": int(parameter_to_index[parameter]),
                    "parameter_name": str(parameter),
                    "primitive_operation": operation_name,
                    "primitive_qubits": qubits,
                    "instruction_position": int(instruction_position),
                    "parameter_slot": int(parameter_slot),
                    "parameter_coefficient": coefficient,
                })

    occurrence_df = pd.DataFrame(occurrence_rows)
    rows = []
    for theta_index, group in occurrence_df.groupby("theta_index"):
        operations = sorted(set(group["primitive_operation"].astype(str)))
        touched_qubits = sorted({
            int(qubit) for qubit_tuple in group["primitive_qubits"] for qubit in qubit_tuple
        })
        if "cry" in operations:
            primitive_type = "CRY"
            angular_period = float(4 * np.pi)
            periodicity_class = "four_pi_eligible"
        else:
            primitive_type = "RY"
            angular_period = float(2 * np.pi)
            periodicity_class = "guaranteed_2pi"
        rows.append({
            "theta_index": int(theta_index),
            "primitive_physical_type": primitive_type,
            "primitive_operations": tuple(operations),
            "physical_qubits": tuple(touched_qubits),
            "angular_period": angular_period,
            "periodicity_structural_class": periodicity_class,
            "n_occurrences_decomposed": int(len(group)),
            "parameter_coefficients": tuple(float(v) for v in group["parameter_coefficient"]),
            "first_instruction": int(group["instruction_position"].min()),
            "last_instruction": int(group["instruction_position"].max()),
            "fourier_n_grid_minimum": 5,
        })

    parameter_map = (
        structure_df.merge(pd.DataFrame(rows), on="theta_index", how="left", validate="one_to_one")
        .sort_values("theta_index")
        .reset_index(drop=True)
    )
    if parameter_map["primitive_physical_type"].isna().any():
        raise RuntimeError("Mapa físico incompleto após a decomposição.")

    cy = parameter_map["ansatz_gate_type"].eq("CY")
    ccy = parameter_map["ansatz_gate_type"].eq("CCY")
    checks = {
        "CY_is_CRY": bool(parameter_map.loc[cy, "primitive_physical_type"].eq("CRY").all()),
        "CY_has_one_occurrence": bool(parameter_map.loc[cy, "n_occurrences_decomposed"].eq(1).all()),
        "CY_has_4pi_max_period": bool(np.allclose(parameter_map.loc[cy, "angular_period"], 4 * np.pi)),
        "CCY_is_RY": bool(parameter_map.loc[ccy, "primitive_physical_type"].eq("RY").all()),
        "CCY_has_two_occurrences": bool(parameter_map.loc[ccy, "n_occurrences_decomposed"].eq(2).all()),
        "CCY_has_2pi_period": bool(np.allclose(parameter_map.loc[ccy, "angular_period"], 2 * np.pi)),
    }
    failed = [name for name, value in checks.items() if not value]
    if failed:
        raise RuntimeError(f"Auditoria estrutural falhou: {failed}")
    return parameter_map, occurrence_df, checks


target_ansatz, target_structure = build_tracked_dicke_ansatz(
    N_ASSETS, TARGET_K, RANDOM_SEED + 100 * N_ASSETS + TARGET_K
)
target_parameter_map, target_occurrences, periodicity_audit = build_physical_parameter_map(
    target_ansatz, target_structure
)

print("parâmetros k=4:", target_ansatz.num_parameters)
print(json.dumps(periodicity_audit, indent=2, ensure_ascii=False))
display(target_parameter_map)


# Parte IV — contextos multi-$k$

Agora a mesma instância financeira é reconstruída para $k=2,3,4,5$. Cada cardinalidade recebe um `problem_id` próprio e mantém juntos:

- enumeração exata e tabela $G_{ij}$;
- QUBO/Ising e offset;
- solução exata;
- ansatz de Dicke;
- mapa físico dos parâmetros;
- simulador e estimador.

Isso impede misturar $\theta$ de um circuito com o Hamiltoniano de outro problema.


In [ ]:
# ============================================================
# 10. CONSTRUIR CONTEXTOS MULTI-k
# ============================================================


def build_problem_context(k_value, classical_analysis):
    encoding = build_docplex_ising(k_value)
    if not np.isclose(
        encoding["exact_energy_ising"], classical_analysis["exact_energy"],
        atol=1e-10, rtol=0.0,
    ):
        raise RuntimeError(f"k={k_value}: Ising e enumeração não coincidem.")
    if not set(classical_analysis["exact_qiskit_bitstrings"]).intersection(
        encoding["exact_bitstrings_ising"]
    ):
        raise RuntimeError(f"k={k_value}: bitstrings Ising e clássicos não coincidem.")

    ansatz, structure_df = build_tracked_dicke_ansatz(
        N_ASSETS, k_value, RANDOM_SEED + 100 * N_ASSETS + k_value
    )
    parameter_map, occurrence_df, structure_audit = build_physical_parameter_map(ansatz, structure_df)
    simulator = AerSimulator(method="statevector", device="CPU")
    estimator_seed = int(RANDOM_SEED + 500_000 + 100 * int(k_value))
    estimator = BackendEstimatorV2(
        backend=simulator,
        options={
            "default_precision": float(ESTIMATOR_PRECISION),
            "abelian_grouping": True,
            "seed_simulator": estimator_seed,
        },
    )
    problem_id = f"{DATA_HASH}_n{N_ASSETS}_k{k_value}_q{Q_VALUE:.3f}"

    return {
        "problem_id": problem_id,
        "hamiltonian_id": DATA_HASH,
        "n": N_ASSETS,
        "k": int(k_value),
        "tickers": tickers,
        "assets_returns": assets_returns.copy(),
        "covariance": covariance.copy(),
        "classical": classical_analysis,
        "model": encoding["model"],
        "qubo": encoding["qubo"],
        "ising": encoding["ising"],
        "offset": encoding["offset"],
        "pauli_labels": encoding["pauli_labels"],
        "exact_energy": classical_analysis["exact_energy"],
        "exact_bitstrings": classical_analysis["exact_qiskit_bitstrings"],
        "ansatz": ansatz,
        "n_parameters": int(ansatz.num_parameters),
        "structure_df": structure_df,
        "parameter_map": parameter_map,
        "occurrence_df": occurrence_df,
        "structure_audit": structure_audit,
        "simulator": simulator,
        "estimator": estimator,
        "estimator_precision": float(ESTIMATOR_PRECISION),
        "estimator_shots_equivalent": int(SHOTS),
        "estimator_seed": estimator_seed,
    }


classical_by_k = {TARGET_K: target_classical}
for k_value in K_VALUES:
    if k_value not in classical_by_k:
        classical_by_k[k_value] = build_classical_analysis(k_value)

problem_contexts = {
    k_value: build_problem_context(k_value, classical_by_k[k_value])
    for k_value in K_VALUES
}

context_summary_df = pd.DataFrame([
    {
        "problem_id": context["problem_id"],
        "hamiltonian_id": context["hamiltonian_id"],
        "n": context["n"],
        "k": context["k"],
        "n_valid_portfolios": len(context["classical"]["enumeration_df"]),
        "n_parameters": context["n_parameters"],
        "exact_energy": context["exact_energy"],
        "exact_bitstrings": context["exact_bitstrings"],
        "selected_assets": context["classical"]["selected_assets"],
        "largest_G_pair": context["classical"]["pair_table"].iloc[0]["pair"],
        "largest_G": float(context["classical"]["pair_table"].iloc[0]["G_ij"]),
    }
    for context in problem_contexts.values()
]).sort_values("k")

display(context_summary_df)


## Auditoria do subespaço de Dicke

Antes de gerar bancos, o circuito é testado com um vetor aleatório por cardinalidade. Toda a probabilidade deve permanecer no subespaço de peso de Hamming $k$. Essa auditoria verifica diretamente a propriedade que justifica usar o ansatz de Dicke para seleção de portfólio com cardinalidade fixa.


In [ ]:
# ============================================================
# 10.1 AUDITORIA DE PRESERVAÇÃO DA CARDINALIDADE
# ============================================================

cardinality_audit_rows = []
for k_value, context in problem_contexts.items():
    local_rng = np.random.default_rng(RANDOM_SEED + 90_000 + k_value)
    theta_test = local_rng.uniform(-np.pi, np.pi, size=context["n_parameters"])
    assigned = context["ansatz"].assign_parameters(theta_test, inplace=False)
    probabilities = Statevector.from_instruction(assigned).probabilities_dict()
    leakage = float(sum(
        probability
        for bitstring, probability in probabilities.items()
        if str(bitstring).replace(" ", "").count("1") != k_value
    ))
    cardinality_audit_rows.append({
        "problem_id": context["problem_id"],
        "k": int(k_value),
        "probability_outside_hamming_weight_k": leakage,
        "cardinality_preserved": bool(leakage <= 1e-10),
    })

cardinality_audit_df = pd.DataFrame(cardinality_audit_rows).sort_values("k")
if not cardinality_audit_df["cardinality_preserved"].all():
    raise RuntimeError("O ansatz apresentou vazamento para fora do subespaço de Dicke.")

display(cardinality_audit_df)


# Parte V — banco definitivo de parâmetros $\theta$

Esta etapa usa apenas um desenho:

- 100 reinicializações por cardinalidade;
- 250 avaliações COBYLA por reinicialização;
- nenhum aquecimento de subconjunto;
- avaliação final exata por `Statevector`;
- amostragem de 4096 shots mantida apenas como observável adicional.

`GENERATE_OR_RESUME_BANKS=False` não cria uma versão piloto. Apenas permite executar as partes anteriores sem iniciar automaticamente 400 otimizações. Ao ativar a flag, o notebook retoma checkpoints e completa exatamente o banco definido acima.


In [ ]:
# ============================================================
# 11. MÉTRICAS EXATAS E UMA RODADA COBYLA
# ============================================================


def exact_theta_metrics(context, theta):
    theta = np.asarray(theta, dtype=float).reshape(-1)
    if len(theta) != context["n_parameters"]:
        raise ValueError(
            f"Vetor theta com {len(theta)} elementos; esperados {context['n_parameters']}."
        )
    assigned = context["ansatz"].assign_parameters(theta, inplace=False)
    state = Statevector.from_instruction(assigned)
    energy = float(np.real(state.expectation_value(context["ising"])) + context["offset"])
    probabilities = {
        str(key).replace(" ", ""): float(value)
        for key, value in state.probabilities_dict().items()
    }
    p_opt = float(sum(probabilities.get(bitstring, 0.0) for bitstring in context["exact_bitstrings"]))
    dominant = max(probabilities, key=probabilities.get)
    return {
        "energy_exact_eval": energy,
        "gap_exact_eval": abs(energy - context["exact_energy"]),
        "p_exact_eval": p_opt,
        "dominant_bitstring_exact_eval": dominant,
        "is_exact_dominant_eval": bool(dominant in set(context["exact_bitstrings"])),
    }


def run_cobyla_bank_row(context, run_index):
    local_rng = np.random.default_rng(
        RANDOM_SEED + 1_000_000 + 10_000 * context["n"] + 100 * context["k"] + int(run_index)
    )
    theta_initial = 0.5 * np.pi * local_rng.random(context["n_parameters"])
    history = []

    def objective(theta):
        theta = np.asarray(theta, dtype=float).reshape(-1)
        result = context["estimator"].run(
            pubs=[(context["ansatz"], context["ising"], theta)],
            precision=context["estimator_precision"],
        ).result()
        energy = float(
            np.real(np.asarray(result[0].data.evs).reshape(-1)[0]) + context["offset"]
        )
        history.append((theta.copy(), energy))
        return energy

    start = perf_counter()
    result = COBYLA(maxiter=COBYLA_MAXITER).minimize(fun=objective, x0=theta_initial)
    optimizer_time = perf_counter() - start
    theta_final = np.asarray(result.x, dtype=float).reshape(-1)
    exact_metrics = exact_theta_metrics(context, theta_final)

    measured = context["ansatz"].assign_parameters(theta_final, inplace=False).copy()
    measured.measure_all()
    simulator_start = perf_counter()
    counts = (
        context["simulator"].run(
            measured,
            shots=SHOTS,
            seed_simulator=RANDOM_SEED + 2_000_000 + 10_000 * context["n"] + 100 * context["k"] + int(run_index),
        ).result().get_counts()
    )
    simulator_time = perf_counter() - simulator_start
    counts = {str(key).replace(" ", ""): int(value) for key, value in counts.items()}
    total_counts = max(sum(counts.values()), 1)
    most_frequent = max(counts, key=counts.get)
    p_opt_shots = float(sum(counts.get(bit, 0) for bit in context["exact_bitstrings"]) / total_counts)

    return {
        "problem_id": context["problem_id"],
        "hamiltonian_id": context["hamiltonian_id"],
        "n": context["n"],
        "k": context["k"],
        "run": int(run_index),
        "n_parameters": context["n_parameters"],
        "initial_point": theta_initial,
        "best_parameters": theta_final,
        "objective_function_value": float(result.fun),
        "exact_energy": context["exact_energy"],
        "nfev": int(len(history)),
        "optimizer_time": float(optimizer_time),
        "simulator_time": float(simulator_time),
        "shots": int(SHOTS),
        "estimator_precision": float(context["estimator_precision"]),
        "estimator_shots_equivalent": int(context["estimator_shots_equivalent"]),
        "probability_best_answer_shots": p_opt_shots,
        "most_frequent_bitstring": most_frequent,
        "is_exact_dominant_shots": bool(most_frequent in set(context["exact_bitstrings"])),
        "counts": counts,
        **exact_metrics,
        "status": "ok",
    }


In [ ]:
# ============================================================
# 12. GERAR, RETOMAR OU CARREGAR OS BANCOS
# ============================================================

bank_frames = {}
bank_status_rows = []

for k_value, context in problem_contexts.items():
    config_dir = bank_dir / f"n{N_ASSETS}k{k_value}"
    config_dir.mkdir(parents=True, exist_ok=True)
    checkpoint_path = config_dir / "bank_rows.pkl"
    final_path = config_dir / "theta_bank.pkl"

    if checkpoint_path.exists():
        rows = pd.read_pickle(checkpoint_path)
        if not isinstance(rows, list):
            rows = []
    elif final_path.exists():
        existing_df = pd.read_pickle(final_path)
        rows = existing_df.to_dict("records")
    else:
        rows = []

    completed = {
        int(row["run"]) for row in rows
        if row.get("status") == "ok"
    }

    if GENERATE_OR_RESUME_BANKS:
        for run_index in range(BANK_RUNS_PER_CONFIGURATION):
            if run_index in completed:
                continue
            row = run_cobyla_bank_row(context, run_index)
            rows.append(row)
            pd.to_pickle(rows, checkpoint_path)
            print(
                f"k={k_value} run {run_index + 1}/{BANK_RUNS_PER_CONFIGURATION} "
                f"gap={row['gap_exact_eval']:.6e} p*={row['p_exact_eval']:.4f}"
            )

    bank_df = pd.DataFrame(rows)
    if not bank_df.empty:
        bank_df = (
            bank_df.loc[bank_df["status"].eq("ok")]
            .drop_duplicates(subset=["problem_id", "run"], keep="last")
            .sort_values("run")
            .reset_index(drop=True)
        )
        bank_df.to_pickle(final_path)
    bank_frames[k_value] = bank_df
    bank_status_rows.append({
        "problem_id": context["problem_id"],
        "k": k_value,
        "target_rows": BANK_RUNS_PER_CONFIGURATION,
        "available_rows": int(len(bank_df)),
        "complete": bool(len(bank_df) >= BANK_RUNS_PER_CONFIGURATION),
        "path": str(final_path.resolve()),
    })

bank_status_df = pd.DataFrame(bank_status_rows).sort_values("k")
display(bank_status_df)

if not GENERATE_OR_RESUME_BANKS and not bank_status_df["complete"].all():
    print(
        "Os bancos ainda não estão completos. A infraestrutura e as tabelas clássicas/quânticas "
        "continuam válidas; ative GENERATE_OR_RESUME_BANKS para completar a única amostra definida."
    )


# Parte VI — experimento com $\theta_{17}$ fixado e reotimização dos demais parâmetros

Este teste responde a uma pergunta diferente da intervenção causal anterior:

> Se $\theta_{17}$ for mantido numa região angular associada às melhores soluções, o COBYLA consegue reorganizar os outros 29 parâmetros e preservar uma distribuição de probabilidade favorável?

O experimento é executado somente no problema original $k=4$ e usa o banco definitivo já produzido. A região ótima não é escolhida manualmente:

1. selecionam-se as execuções simultaneamente no quartil superior de $P(x^*)$ e no quartil inferior do gap;
2. se a interseção for pequena, utiliza-se uma seleção lexicográfica por maior $P(x^*)$ e menor gap;
3. calcula-se o menor arco circular que cobre 80% desses valores de $\theta_{17}$, respeitando o período físico registrado no mapa do ansatz;
4. esse arco é amostrado em sete valores fixos.

Para cada restart, as mesmas coordenadas iniciais dos outros parâmetros são usadas em três condições:

- **`free_baseline`**: todos os parâmetros ficam livres, iniciando com $\theta_{17}$ no centro do arco ótimo;
- **`fixed_optimal_interval`**: $\theta_{17}$ é retirado do vetor do otimizador e permanece fixo num ponto do arco ótimo;
- **`fixed_shifted_control`**: usa-se o mesmo arco deslocado por metade do período como controle estrutural.

O intervalo deslocado não é declarado previamente como “ruim”; ele é um controle de fase de mesma largura.

## Duas distribuições são avaliadas após cada COBYLA

A energia usada durante a otimização é obtida pelo `BackendEstimatorV2` com precisão explicitamente fixada em

$$
\varepsilon = \frac{1}{\sqrt{4096}},
$$

isto é, um orçamento de medição equivalente a 4096 shots por avaliação do Estimator. Depois de o COBYLA terminar, o mesmo circuito final é avaliado de duas maneiras:

1. **Distribuição exata (`Statevector`)**: $P_{\mathrm{exact}}(x)=|\langle x|\psi(	heta)angle|^2$ para todos os 1024 bitstrings;
2. **Distribuição observada com 4096 shots**: $\widehat P_{4096}(x)=N_x/4096$.

Como o ansatz de Dicke preserva $k=4$, a massa física deve permanecer nos 210 bitstrings válidos. O notebook compara as duas distribuições por distância de variação total, divergência de Jensen–Shannon, erro máximo e RMSE. Isso separa o efeito matemático de fixar $	heta_{17}$ da flutuação de medição que seria observada com 4096 shots.

O teste compara energia, $P(x^*)$ exata e amostrada, bitstring dominante, entropia, participação efetiva, distância de variação total e divergência de Jensen–Shannon da distribuição completa.

### Interpretação

- Se a região ótima superar o controle deslocado após a reotimização, existe evidência de que $\theta_{17}$ carrega informação operacional não totalmente compensável pelos demais parâmetros.
- Se o baseline livre vencer com folga, fixar o parâmetro restringe excessivamente o ansatz.
- Se todas as condições forem equivalentes, o COBYLA compensa a restrição e a importância observada de $\theta_{17}$ não se traduz diretamente em vantagem de otimização.


In [ ]:
# ============================================================
# 13. THETA_17 FIXO: COBYLA NOS OUTROS PARÂMETROS
# ============================================================


def canonical_positive_angle(angle, period):
    return float(np.mod(float(angle), float(period)))


def shortest_circular_arc(angles, period, coverage=0.80, minimum_fraction=0.05):
    """
    Menor arco circular que cobre `coverage` das observações.
    Retorna limites desenrolados; valores físicos são obtidos módulo `period`.
    """
    values = np.sort(np.mod(np.asarray(angles, dtype=float), period))
    if len(values) == 0:
        raise ValueError("Não há ângulos para construir o intervalo.")
    n_cover = max(1, int(np.ceil(float(coverage) * len(values))))
    extended = np.concatenate([values, values + period])
    candidates = []
    for start_index in range(len(values)):
        end_index = start_index + n_cover - 1
        width = float(extended[end_index] - extended[start_index])
        candidates.append((width, start_index, end_index))
    width, start_index, end_index = min(candidates, key=lambda item: item[0])
    start = float(extended[start_index])
    end = float(extended[end_index])
    center = 0.5 * (start + end)

    minimum_width = float(minimum_fraction) * float(period)
    was_expanded = bool(width < minimum_width)
    if was_expanded:
        start = center - 0.5 * minimum_width
        end = center + 0.5 * minimum_width
        width = minimum_width

    return {
        "start_unwrapped": float(start),
        "end_unwrapped": float(end),
        "center_unwrapped": float(0.5 * (start + end)),
        "start": canonical_positive_angle(start, period),
        "end": canonical_positive_angle(end, period),
        "center": canonical_positive_angle(0.5 * (start + end), period),
        "width": float(width),
        "coverage": float(coverage),
        "n_input": int(len(values)),
        "n_covered": int(n_cover),
        "minimum_width_applied": was_expanded,
    }


def select_theta17_optimal_rows(bank_df):
    required = {
        "best_parameters",
        "p_exact_eval",
        "gap_exact_eval",
        "status",
    }
    missing = sorted(required - set(bank_df.columns))
    if missing:
        raise RuntimeError(f"Banco k=4 sem colunas necessárias: {missing}")

    valid = (
        bank_df.loc[bank_df["status"].eq("ok")]
        .dropna(subset=["p_exact_eval", "gap_exact_eval"])
        .copy()
    )
    if len(valid) < THETA17_MIN_OPTIMAL_ROWS:
        raise RuntimeError(
            "O experimento de theta_17 requer ao menos "
            f"{THETA17_MIN_OPTIMAL_ROWS} execuções válidas no banco k=4; "
            f"foram encontradas {len(valid)}."
        )

    p_threshold = float(valid["p_exact_eval"].quantile(1.0 - THETA17_OPTIMAL_QUANTILE))
    gap_threshold = float(valid["gap_exact_eval"].quantile(THETA17_OPTIMAL_QUANTILE))
    selected = valid.loc[
        valid["p_exact_eval"].ge(p_threshold)
        & valid["gap_exact_eval"].le(gap_threshold)
    ].copy()

    fallback_used = False
    if len(selected) < THETA17_MIN_OPTIMAL_ROWS:
        fallback_used = True
        selected = (
            valid.sort_values(
                ["p_exact_eval", "gap_exact_eval"],
                ascending=[False, True],
            )
            .head(THETA17_MIN_OPTIMAL_ROWS)
            .copy()
        )

    return selected, {
        "n_valid_bank_rows": int(len(valid)),
        "n_selected_rows": int(len(selected)),
        "p_exact_threshold": p_threshold,
        "gap_threshold": gap_threshold,
        "fallback_lexicographic_used": bool(fallback_used),
    }


def exact_distribution_metrics(context, theta):
    theta = np.asarray(theta, dtype=float).reshape(-1)
    assigned = context["ansatz"].assign_parameters(theta, inplace=False)
    state = Statevector.from_instruction(assigned)
    probabilities = np.asarray(state.probabilities(), dtype=float)
    probabilities = probabilities / max(float(probabilities.sum()), np.finfo(float).eps)
    labels = np.asarray(
        [format(index, f"0{context['n']}b") for index in range(len(probabilities))],
        dtype=object,
    )
    energy = float(np.real(state.expectation_value(context["ising"])) + context["offset"])
    optimal_set = set(context["exact_bitstrings"])
    optimal_mask = np.asarray([label in optimal_set for label in labels], dtype=bool)
    p_opt = float(probabilities[optimal_mask].sum())
    dominant_index = int(np.argmax(probabilities))
    nonzero = probabilities[probabilities > 0.0]
    entropy = float(-np.sum(nonzero * np.log(nonzero)))
    participation_ratio = float(1.0 / np.sum(probabilities ** 2))
    return {
        "probability_vector": probabilities,  # alias legado: distribuição exata
        "probability_vector_exact": probabilities,
        "bitstring_labels": labels,
        "energy_exact_eval": energy,
        "gap_exact_eval": abs(energy - context["exact_energy"]),
        "p_exact_eval": p_opt,
        "dominant_bitstring_exact_eval": str(labels[dominant_index]),
        "dominant_probability_exact_eval": float(probabilities[dominant_index]),
        "is_exact_dominant_eval": bool(labels[dominant_index] in optimal_set),
        "entropy": entropy,
        "normalized_entropy": float(entropy / np.log(len(probabilities))),
        "participation_ratio": participation_ratio,
    }



def sampled_distribution_metrics(context, theta, shots, seed_simulator):
    """Mede o circuito final e devolve a distribuição discreta observada."""
    theta = np.asarray(theta, dtype=float).reshape(-1)
    assigned = context["ansatz"].assign_parameters(theta, inplace=False)
    measured = assigned.copy()
    measured.measure_all()

    start = perf_counter()
    raw_counts = (
        context["simulator"].run(
            measured,
            shots=int(shots),
            seed_simulator=int(seed_simulator),
        )
        .result()
        .get_counts()
    )
    elapsed = perf_counter() - start

    counts = {
        str(bitstring).replace(" ", ""): int(count)
        for bitstring, count in raw_counts.items()
    }
    labels = np.asarray(
        [format(index, f"0{context['n']}b") for index in range(2 ** context["n"])],
        dtype=object,
    )
    total = max(int(sum(counts.values())), 1)
    probabilities = np.asarray(
        [counts.get(str(label), 0) / total for label in labels],
        dtype=float,
    )
    probabilities = probabilities / max(
        float(probabilities.sum()), np.finfo(float).eps
    )

    optimal_set = set(context["exact_bitstrings"])
    optimal_mask = np.asarray([label in optimal_set for label in labels], dtype=bool)
    p_opt = float(probabilities[optimal_mask].sum())
    dominant_index = int(np.argmax(probabilities))
    nonzero = probabilities[probabilities > 0.0]
    entropy = float(-np.sum(nonzero * np.log(nonzero)))
    participation_ratio = float(1.0 / np.sum(probabilities ** 2))

    return {
        "probability_vector_shots": probabilities,
        "counts_shots": counts,
        "distribution_shots": int(total),
        "measurement_seed": int(seed_simulator),
        "simulator_time_shots": float(elapsed),
        "p_opt_shots": p_opt,
        "dominant_bitstring_shots": str(labels[dominant_index]),
        "dominant_probability_shots": float(probabilities[dominant_index]),
        "is_exact_dominant_shots": bool(labels[dominant_index] in optimal_set),
        "entropy_shots": entropy,
        "normalized_entropy_shots": float(entropy / np.log(len(probabilities))),
        "participation_ratio_shots": participation_ratio,
    }


def total_variation_distance(p, q):
    p = np.asarray(p, dtype=float)
    q = np.asarray(q, dtype=float)
    return float(0.5 * np.abs(p - q).sum())


def jensen_shannon_divergence(p, q):
    p = np.asarray(p, dtype=float)
    q = np.asarray(q, dtype=float)
    midpoint = 0.5 * (p + q)

    def kl_divergence(a, b):
        mask = a > 0.0
        return float(np.sum(a[mask] * np.log(a[mask] / b[mask])))

    return float(
        0.5 * kl_divergence(p, midpoint)
        + 0.5 * kl_divergence(q, midpoint)
    )


def run_cobyla_with_theta_constraint(
    context,
    initial_theta,
    run_index,
    condition,
    fixed_theta_value=np.nan,
):
    initial_theta = np.asarray(initial_theta, dtype=float).reshape(-1).copy()
    if len(initial_theta) != context["n_parameters"]:
        raise ValueError("Dimensão do ponto inicial incompatível com o ansatz.")

    is_fixed = condition != "free_baseline"
    free_indices = np.asarray(
        [index for index in range(context["n_parameters"]) if index != THETA17_INDEX],
        dtype=int,
    )
    history = []

    if is_fixed:
        theta_fixed = float(fixed_theta_value)
        initial_theta[THETA17_INDEX] = theta_fixed
        x0 = initial_theta[free_indices].copy()

        def unpack(candidate):
            full = initial_theta.copy()
            full[free_indices] = np.asarray(candidate, dtype=float)
            full[THETA17_INDEX] = theta_fixed
            return full
    else:
        x0 = initial_theta.copy()

        def unpack(candidate):
            return np.asarray(candidate, dtype=float).reshape(-1)

    def objective(candidate):
        theta_full = unpack(candidate)
        result = context["estimator"].run(
            pubs=[(context["ansatz"], context["ising"], theta_full)],
            precision=context["estimator_precision"],
        ).result()
        energy = float(
            np.real(np.asarray(result[0].data.evs).reshape(-1)[0])
            + context["offset"]
        )
        history.append(energy)
        return energy

    start = perf_counter()
    result = COBYLA(maxiter=COBYLA_MAXITER).minimize(fun=objective, x0=x0)
    elapsed = perf_counter() - start
    theta_final = unpack(result.x)
    metrics = exact_distribution_metrics(context, theta_final)

    seed_payload = (
        f"{RANDOM_SEED}|{context['problem_id']}|{run_index}|{condition}|"
        f"{float(fixed_theta_value) if is_fixed else 'free'}|{THETA17_DISTRIBUTION_SHOTS}"
    )
    measurement_seed = int(
        hashlib.sha256(seed_payload.encode("utf-8")).hexdigest()[:8], 16
    ) % (2**31 - 1)
    sampled_metrics = sampled_distribution_metrics(
        context=context,
        theta=theta_final,
        shots=THETA17_DISTRIBUTION_SHOTS,
        seed_simulator=measurement_seed,
    )
    sampled_metrics["tvd_shots_vs_exact"] = total_variation_distance(
        sampled_metrics["probability_vector_shots"],
        metrics["probability_vector_exact"],
    )
    sampled_metrics["jsd_shots_vs_exact"] = jensen_shannon_divergence(
        sampled_metrics["probability_vector_shots"],
        metrics["probability_vector_exact"],
    )
    residual = (
        sampled_metrics["probability_vector_shots"]
        - metrics["probability_vector_exact"]
    )
    sampled_metrics["max_abs_probability_error_shots"] = float(
        np.max(np.abs(residual))
    )
    sampled_metrics["rmse_probability_shots"] = float(
        np.sqrt(np.mean(residual ** 2))
    )
    sampled_metrics["p_opt_shots_minus_exact"] = float(
        sampled_metrics["p_opt_shots"] - metrics["p_exact_eval"]
    )

    if is_fixed and not np.isclose(
        theta_final[THETA17_INDEX],
        float(fixed_theta_value),
        atol=1e-12,
        rtol=0.0,
    ):
        raise RuntimeError("theta_17 deixou de permanecer fixo durante a otimização.")

    return {
        "problem_id": context["problem_id"],
        "hamiltonian_id": context["hamiltonian_id"],
        "n": context["n"],
        "k": context["k"],
        "restart": int(run_index),
        "condition": str(condition),
        "fixed_theta_index": int(THETA17_INDEX),
        "fixed_theta_value": (
            float(fixed_theta_value) if is_fixed else np.nan
        ),
        "initial_theta": initial_theta,
        "best_parameters": theta_final,
        "theta17_final": float(theta_final[THETA17_INDEX]),
        "objective_function_value": float(result.fun),
        "nfev": int(len(history)),
        "optimizer_time": float(elapsed),
        "estimator_precision": float(context["estimator_precision"]),
        "estimator_shots_equivalent": int(context["estimator_shots_equivalent"]),
        "status": "ok",
        **metrics,
        **sampled_metrics,
    }


theta17_runs_df = pd.DataFrame()
theta17_summary_df = pd.DataFrame()
theta17_probability_long_df = pd.DataFrame()
theta17_paired_tests_df = pd.DataFrame()
theta17_interval_metadata = {
    "enabled": bool(RUN_THETA17_FIXED_EXPERIMENT),
    "executed": False,
}

if RUN_THETA17_FIXED_EXPERIMENT:
    context = problem_contexts[TARGET_K]
    bank_df = bank_frames[TARGET_K]
    if bank_df.empty:
        raise RuntimeError(
            "O banco k=4 está vazio. Gere ou carregue o banco definitivo antes "
            "de executar o experimento com theta_17 fixo."
        )
    if THETA17_INDEX >= context["n_parameters"]:
        raise RuntimeError(
            f"theta_{THETA17_INDEX} não existe no ansatz k={TARGET_K}."
        )

    parameter_row = (
        context["parameter_map"]
        .loc[lambda df: df["theta_index"].eq(THETA17_INDEX)]
    )
    if len(parameter_row) != 1:
        raise RuntimeError("O mapa físico não identificou theta_17 de forma única.")
    parameter_row = parameter_row.iloc[0]
    theta_period = float(parameter_row["angular_period"])

    selected_rows, selection_metadata = select_theta17_optimal_rows(bank_df)
    theta17_optimal_values = np.asarray(
        [
            np.asarray(theta, dtype=float)[THETA17_INDEX]
            for theta in selected_rows["best_parameters"]
        ],
        dtype=float,
    )
    interval = shortest_circular_arc(
        theta17_optimal_values,
        period=theta_period,
        coverage=THETA17_INTERVAL_COVERAGE,
        minimum_fraction=THETA17_MIN_INTERVAL_FRACTION,
    )

    optimal_grid_unwrapped = np.linspace(
        interval["start_unwrapped"],
        interval["end_unwrapped"],
        THETA17_GRID_POINTS,
    )
    optimal_grid = np.mod(optimal_grid_unwrapped, theta_period)
    shifted_grid_unwrapped = optimal_grid_unwrapped + 0.5 * theta_period
    shifted_grid = np.mod(shifted_grid_unwrapped, theta_period)

    experiment_signature_payload = {
        "problem_id": context["problem_id"],
        "theta_index": int(THETA17_INDEX),
        "optimal_grid": [float(v) for v in optimal_grid],
        "shifted_control_grid": [float(v) for v in shifted_grid],
        "restarts": int(THETA17_RESTARTS),
        "cobyla_maxiter": int(COBYLA_MAXITER),
        "distribution_shots": int(THETA17_DISTRIBUTION_SHOTS),
        "estimator_precision": float(ESTIMATOR_PRECISION),
        "schema_version": "exact_plus_shots_v1",
    }
    experiment_signature = hashlib.sha256(
        json.dumps(
            experiment_signature_payload,
            sort_keys=True,
            ensure_ascii=False,
        ).encode("utf-8")
    ).hexdigest()[:12]

    theta17_interval_metadata = {
        "enabled": True,
        "executed": True,
        "experiment_signature": experiment_signature,
        "theta_index": int(THETA17_INDEX),
        "ansatz_gate_type": str(parameter_row["ansatz_gate_type"]),
        "primitive_physical_type": str(parameter_row["primitive_physical_type"]),
        "physical_qubits": tuple(int(v) for v in parameter_row["physical_qubits"]),
        "angular_period": theta_period,
        "selection": selection_metadata,
        "interval": interval,
        "optimal_grid": [float(v) for v in optimal_grid],
        "shifted_control_grid": [float(v) for v in shifted_grid],
        "restarts": int(THETA17_RESTARTS),
        "cobyla_maxiter": int(COBYLA_MAXITER),
        "distribution_shots": int(THETA17_DISTRIBUTION_SHOTS),
        "estimator_precision": float(ESTIMATOR_PRECISION),
        "distribution_outputs": ["statevector_exact", "sampled_4096_shots"],
    }

    checkpoint_path = (
        theta17_dir
        / f"theta17_fixed_rows_{experiment_signature}.pkl"
    )
    if checkpoint_path.exists():
        checkpoint_rows = pd.read_pickle(checkpoint_path)
        rows = checkpoint_rows if isinstance(checkpoint_rows, list) else []
    else:
        rows = []

    completed = {
        (
            str(row["condition"]),
            int(row["restart"]),
            int(row.get("grid_index", -1)),
        )
        for row in rows
        if row.get("status") == "ok"
    }

    for restart in range(THETA17_RESTARTS):
        rng = np.random.default_rng(
            RANDOM_SEED + 7_000_000 + 1000 * TARGET_K + restart
        )
        common_initial = 0.5 * np.pi * rng.random(context["n_parameters"])
        common_initial[THETA17_INDEX] = float(interval["center"])

        baseline_key = ("free_baseline", restart, -1)
        if baseline_key not in completed:
            row = run_cobyla_with_theta_constraint(
                context=context,
                initial_theta=common_initial,
                run_index=restart,
                condition="free_baseline",
            )
            row["grid_index"] = -1
            row["grid_position"] = np.nan
            row["fixed_theta_unwrapped"] = np.nan
            rows.append(row)
            pd.to_pickle(rows, checkpoint_path)
            completed.add(baseline_key)
            print(
                f"theta17 baseline restart={restart + 1}/{THETA17_RESTARTS} "
                f"p*={row['p_exact_eval']:.4f} gap={row['gap_exact_eval']:.3e}"
            )

        for grid_index, (value, unwrapped) in enumerate(
            zip(optimal_grid, optimal_grid_unwrapped)
        ):
            key = ("fixed_optimal_interval", restart, grid_index)
            if key in completed:
                continue
            row = run_cobyla_with_theta_constraint(
                context=context,
                initial_theta=common_initial,
                run_index=restart,
                condition="fixed_optimal_interval",
                fixed_theta_value=float(value),
            )
            row["grid_index"] = int(grid_index)
            row["grid_position"] = float(
                grid_index / max(THETA17_GRID_POINTS - 1, 1)
            )
            row["fixed_theta_unwrapped"] = float(unwrapped)
            rows.append(row)
            pd.to_pickle(rows, checkpoint_path)
            completed.add(key)

        for grid_index, (value, unwrapped) in enumerate(
            zip(shifted_grid, shifted_grid_unwrapped)
        ):
            key = ("fixed_shifted_control", restart, grid_index)
            if key in completed:
                continue
            row = run_cobyla_with_theta_constraint(
                context=context,
                initial_theta=common_initial,
                run_index=restart,
                condition="fixed_shifted_control",
                fixed_theta_value=float(value),
            )
            row["grid_index"] = int(grid_index)
            row["grid_position"] = float(
                grid_index / max(THETA17_GRID_POINTS - 1, 1)
            )
            row["fixed_theta_unwrapped"] = float(unwrapped)
            rows.append(row)
            pd.to_pickle(rows, checkpoint_path)
            completed.add(key)

        print(
            f"theta17 restart {restart + 1}/{THETA17_RESTARTS} concluído."
        )

    theta17_runs_df = (
        pd.DataFrame(rows)
        .loc[lambda df: df["status"].eq("ok")]
        .drop_duplicates(
            subset=["condition", "restart", "grid_index"],
            keep="last",
        )
        .sort_values(["condition", "grid_index", "restart"])
        .reset_index(drop=True)
    )

    baseline_vectors_exact = {
        int(row["restart"]): np.asarray(row["probability_vector_exact"], dtype=float)
        for _, row in theta17_runs_df.loc[
            theta17_runs_df["condition"].eq("free_baseline")
        ].iterrows()
    }
    baseline_vectors_shots = {
        int(row["restart"]): np.asarray(row["probability_vector_shots"], dtype=float)
        for _, row in theta17_runs_df.loc[
            theta17_runs_df["condition"].eq("free_baseline")
        ].iterrows()
    }
    theta17_runs_df["tvd_exact_vs_free_baseline"] = theta17_runs_df.apply(
        lambda row: (
            0.0
            if row["condition"] == "free_baseline"
            else total_variation_distance(
                row["probability_vector_exact"],
                baseline_vectors_exact[int(row["restart"])],
            )
        ),
        axis=1,
    )
    theta17_runs_df["jsd_exact_vs_free_baseline"] = theta17_runs_df.apply(
        lambda row: (
            0.0
            if row["condition"] == "free_baseline"
            else jensen_shannon_divergence(
                row["probability_vector_exact"],
                baseline_vectors_exact[int(row["restart"])],
            )
        ),
        axis=1,
    )
    theta17_runs_df["tvd_shots_vs_free_baseline"] = theta17_runs_df.apply(
        lambda row: (
            0.0
            if row["condition"] == "free_baseline"
            else total_variation_distance(
                row["probability_vector_shots"],
                baseline_vectors_shots[int(row["restart"])],
            )
        ),
        axis=1,
    )
    theta17_runs_df["jsd_shots_vs_free_baseline"] = theta17_runs_df.apply(
        lambda row: (
            0.0
            if row["condition"] == "free_baseline"
            else jensen_shannon_divergence(
                row["probability_vector_shots"],
                baseline_vectors_shots[int(row["restart"])],
            )
        ),
        axis=1,
    )
    # aliases legados passam a representar a distribuição exata.
    theta17_runs_df["tvd_vs_free_baseline"] = theta17_runs_df[
        "tvd_exact_vs_free_baseline"
    ]
    theta17_runs_df["jsd_vs_free_baseline"] = theta17_runs_df[
        "jsd_exact_vs_free_baseline"
    ]

    theta17_summary_df = (
        theta17_runs_df.groupby(
            ["condition", "grid_index", "fixed_theta_value"],
            dropna=False,
        )
        .agg(
            n=("restart", "nunique"),
            p_exact_median=("p_exact_eval", "median"),
            p_exact_q25=("p_exact_eval", lambda s: float(s.quantile(0.25))),
            p_exact_q75=("p_exact_eval", lambda s: float(s.quantile(0.75))),
            p_shots_median=("p_opt_shots", "median"),
            p_shots_q25=("p_opt_shots", lambda s: float(s.quantile(0.25))),
            p_shots_q75=("p_opt_shots", lambda s: float(s.quantile(0.75))),
            p_shots_minus_exact_median=("p_opt_shots_minus_exact", "median"),
            gap_median=("gap_exact_eval", "median"),
            gap_q25=("gap_exact_eval", lambda s: float(s.quantile(0.25))),
            gap_q75=("gap_exact_eval", lambda s: float(s.quantile(0.75))),
            entropy_median=("normalized_entropy", "median"),
            entropy_shots_median=("normalized_entropy_shots", "median"),
            participation_ratio_median=("participation_ratio", "median"),
            participation_ratio_shots_median=("participation_ratio_shots", "median"),
            tvd_median=("tvd_exact_vs_free_baseline", "median"),
            jsd_median=("jsd_exact_vs_free_baseline", "median"),
            tvd_shots_vs_baseline_median=("tvd_shots_vs_free_baseline", "median"),
            jsd_shots_vs_baseline_median=("jsd_shots_vs_free_baseline", "median"),
            tvd_shots_vs_exact_median=("tvd_shots_vs_exact", "median"),
            jsd_shots_vs_exact_median=("jsd_shots_vs_exact", "median"),
            max_abs_probability_error_shots_median=("max_abs_probability_error_shots", "median"),
            rmse_probability_shots_median=("rmse_probability_shots", "median"),
            exact_dominant_rate=("is_exact_dominant_eval", "mean"),
            shots_dominant_rate=("is_exact_dominant_shots", "mean"),
            nfev_median=("nfev", "median"),
            optimizer_time_median=("optimizer_time", "median"),
        )
        .reset_index()
    )

    # Comparações pareadas usam uma média por restart sobre a grade,
    # evitando tratar os sete pontos como réplicas independentes.
    per_restart = (
        theta17_runs_df.groupby(["condition", "restart"])
        .agg(
            p_exact=("p_exact_eval", "mean"),
            p_shots=("p_opt_shots", "mean"),
            gap=("gap_exact_eval", "mean"),
            exact_dominant_rate=("is_exact_dominant_eval", "mean"),
            shots_dominant_rate=("is_exact_dominant_shots", "mean"),
            entropy_exact=("normalized_entropy", "mean"),
            entropy_shots=("normalized_entropy_shots", "mean"),
            tvd_exact=("tvd_exact_vs_free_baseline", "mean"),
            tvd_shots=("tvd_shots_vs_free_baseline", "mean"),
            tvd_shots_vs_exact=("tvd_shots_vs_exact", "mean"),
        )
        .reset_index()
    )
    paired_wide = per_restart.pivot(
        index="restart",
        columns="condition",
    )

    def paired_wilcoxon(metric, left, right):
        left_values = paired_wide[(metric, left)].to_numpy(dtype=float)
        right_values = paired_wide[(metric, right)].to_numpy(dtype=float)
        difference = left_values - right_values
        try:
            statistic, p_value = wilcoxon(
                left_values,
                right_values,
                alternative="two-sided",
                zero_method="wilcox",
            )
        except ValueError:
            statistic, p_value = np.nan, 1.0
        return {
            "metric": metric,
            "left_condition": left,
            "right_condition": right,
            "n_pairs": int(len(difference)),
            "median_left": float(np.median(left_values)),
            "median_right": float(np.median(right_values)),
            "median_difference_left_minus_right": float(np.median(difference)),
            "wilcoxon_statistic": float(statistic) if np.isfinite(statistic) else np.nan,
            "wilcoxon_p_value": float(p_value),
        }

    paired_rows = []
    for metric in [
        "p_exact",
        "p_shots",
        "gap",
        "exact_dominant_rate",
        "shots_dominant_rate",
        "entropy_exact",
        "entropy_shots",
        "tvd_shots_vs_exact",
    ]:
        paired_rows.append(
            paired_wilcoxon(
                metric,
                "fixed_optimal_interval",
                "fixed_shifted_control",
            )
        )
        paired_rows.append(
            paired_wilcoxon(
                metric,
                "fixed_optimal_interval",
                "free_baseline",
            )
        )
    theta17_paired_tests_df = pd.DataFrame(paired_rows)

    # Distribuição média completa por condição e ponto da grade:
    # Statevector exato, 4096 shots e resíduo amostral.
    distribution_rows = []
    for (condition, grid_index), group in theta17_runs_df.groupby(
        ["condition", "grid_index"],
        dropna=False,
    ):
        mean_probability_exact = np.mean(
            np.stack(group["probability_vector_exact"].to_list()),
            axis=0,
        )
        mean_probability_shots = np.mean(
            np.stack(group["probability_vector_shots"].to_list()),
            axis=0,
        )
        labels = group.iloc[0]["bitstring_labels"]
        fixed_value = group["fixed_theta_value"].iloc[0]
        for bitstring, probability_exact, probability_shots in zip(
            labels,
            mean_probability_exact,
            mean_probability_shots,
        ):
            distribution_rows.append({
                "condition": condition,
                "grid_index": int(grid_index),
                "fixed_theta_value": (
                    float(fixed_value) if pd.notna(fixed_value) else np.nan
                ),
                "bitstring": str(bitstring),
                "mean_probability": float(probability_exact),  # alias legado
                "mean_probability_exact": float(probability_exact),
                "mean_probability_shots": float(probability_shots),
                "mean_count_shots": float(
                    probability_shots * THETA17_DISTRIBUTION_SHOTS
                ),
                "mean_residual_shots_minus_exact": float(
                    probability_shots - probability_exact
                ),
                "is_exact_bitstring": bool(
                    bitstring in set(context["exact_bitstrings"])
                ),
                "is_valid_dicke_bitstring": bool(
                    str(bitstring).count("1") == TARGET_K
                ),
            })
    theta17_probability_long_df = pd.DataFrame(distribution_rows)

    display(theta17_interval_metadata)
    display(theta17_summary_df)
    display(theta17_paired_tests_df)

    # ---------- Figuras ----------
    baseline_p_exact = float(
        theta17_runs_df.loc[
            theta17_runs_df["condition"].eq("free_baseline"),
            "p_exact_eval",
        ].median()
    )
    baseline_p_shots = float(
        theta17_runs_df.loc[
            theta17_runs_df["condition"].eq("free_baseline"),
            "p_opt_shots",
        ].median()
    )
    fixed_summary = theta17_summary_df.loc[
        theta17_summary_df["condition"].ne("free_baseline")
    ].copy()

    fig, ax = plt.subplots(figsize=(10, 5))
    for condition, label in [
        ("fixed_optimal_interval", "intervalo ótimo"),
        ("fixed_shifted_control", "controle deslocado T/2"),
    ]:
        subset = fixed_summary.loc[
            fixed_summary["condition"].eq(condition)
        ].sort_values("grid_index")
        ax.plot(
            subset["grid_index"],
            subset["p_exact_median"],
            marker="o",
            label=f"{label} — Statevector",
        )
        ax.plot(
            subset["grid_index"],
            subset["p_shots_median"],
            marker="x",
            linestyle="--",
            label=f"{label} — {THETA17_DISTRIBUTION_SHOTS} shots",
        )
    ax.axhline(
        baseline_p_exact,
        linestyle=":",
        label="baseline livre — Statevector",
    )
    ax.axhline(
        baseline_p_shots,
        linestyle="-.",
        label=f"baseline livre — {THETA17_DISTRIBUTION_SHOTS} shots",
    )
    ax.set_xlabel("posição na grade angular")
    ax.set_ylabel("probabilidade do ótimo P(x*)")
    ax.set_title(
        "Theta 17 fixo: P(x*) exata e observada após reotimização"
    )
    ax.legend()
    fig.tight_layout()
    fig.savefig(figure_dir / "theta17_fixed_probability_exact_vs_shots.png", dpi=180)
    plt.show()

    baseline_gap = float(
        theta17_runs_df.loc[
            theta17_runs_df["condition"].eq("free_baseline"),
            "gap_exact_eval",
        ].median()
    )
    fig, ax = plt.subplots(figsize=(9, 5))
    for condition, label in [
        ("fixed_optimal_interval", "theta17 no intervalo ótimo"),
        ("fixed_shifted_control", "controle deslocado por T/2"),
    ]:
        subset = fixed_summary.loc[
            fixed_summary["condition"].eq(condition)
        ].sort_values("grid_index")
        ax.plot(
            subset["grid_index"],
            subset["gap_median"],
            marker="o",
            label=label,
        )
    ax.axhline(baseline_gap, linestyle="--", label="baseline livre (mediana)")
    ax.set_xlabel("posição na grade angular")
    ax.set_ylabel("gap energético")
    ax.set_title("Theta 17 fixo: gap após reotimizar os demais parâmetros")
    ax.legend()
    fig.tight_layout()
    fig.savefig(figure_dir / "theta17_fixed_gap.png", dpi=180)
    plt.show()

    # Seleciona bitstrings com maior probabilidade em qualquer uma das duas leituras.
    top_bitstrings = (
        theta17_probability_long_df.groupby("bitstring")[[
            "mean_probability_exact",
            "mean_probability_shots",
        ]]
        .max()
        .max(axis=1)
        .nlargest(12)
        .index
        .tolist()
    )

    def build_distribution_heatmap(value_column):
        frame = (
            theta17_probability_long_df.loc[
                theta17_probability_long_df["bitstring"].isin(top_bitstrings)
            ]
            .assign(
                row_label=lambda df: df.apply(
                    lambda row: (
                        "free"
                        if row["condition"] == "free_baseline"
                        else (
                            f"opt_{int(row['grid_index'])}"
                            if row["condition"] == "fixed_optimal_interval"
                            else f"ctrl_{int(row['grid_index'])}"
                        )
                    ),
                    axis=1,
                )
            )
            .pivot(index="row_label", columns="bitstring", values=value_column)
            .fillna(0.0)
        )
        preferred_order = (
            ["free"]
            + [f"opt_{i}" for i in range(THETA17_GRID_POINTS)]
            + [f"ctrl_{i}" for i in range(THETA17_GRID_POINTS)]
        )
        return frame.reindex(
            [label for label in preferred_order if label in frame.index]
        )

    for value_column, title, filename, colorbar_label in [
        (
            "mean_probability_exact",
            "Distribuição média exata — Statevector",
            "theta17_probability_heatmap_exact.png",
            "probabilidade média exata",
        ),
        (
            "mean_probability_shots",
            f"Distribuição média observada — {THETA17_DISTRIBUTION_SHOTS} shots",
            "theta17_probability_heatmap_4096shots.png",
            "frequência relativa média",
        ),
        (
            "mean_residual_shots_minus_exact",
            f"Resíduo: {THETA17_DISTRIBUTION_SHOTS} shots menos Statevector",
            "theta17_probability_heatmap_residual.png",
            "erro amostral médio",
        ),
    ]:
        heatmap_df = build_distribution_heatmap(value_column)
        fig, ax = plt.subplots(figsize=(13, 7))
        image = ax.imshow(heatmap_df.to_numpy(dtype=float), aspect="auto")
        ax.set_xticks(np.arange(len(heatmap_df.columns)))
        ax.set_xticklabels(heatmap_df.columns, rotation=60, ha="right")
        ax.set_yticks(np.arange(len(heatmap_df.index)))
        ax.set_yticklabels(heatmap_df.index)
        ax.set_xlabel("bitstrings de maior probabilidade")
        ax.set_ylabel("condição / ponto angular")
        ax.set_title(title)
        fig.colorbar(image, ax=ax, label=colorbar_label)
        fig.tight_layout()
        fig.savefig(figure_dir / filename, dpi=180)
        plt.show()

    fig, ax = plt.subplots(figsize=(9, 5))
    for condition, label in [
        ("fixed_optimal_interval", "intervalo ótimo"),
        ("fixed_shifted_control", "controle deslocado T/2"),
    ]:
        subset = fixed_summary.loc[
            fixed_summary["condition"].eq(condition)
        ].sort_values("grid_index")
        ax.plot(
            subset["grid_index"],
            subset["tvd_median"],
            marker="o",
            label=f"{label} — Statevector",
        )
        ax.plot(
            subset["grid_index"],
            subset["tvd_shots_vs_baseline_median"],
            marker="x",
            linestyle="--",
            label=f"{label} — {THETA17_DISTRIBUTION_SHOTS} shots",
        )
    ax.set_xlabel("posição na grade angular")
    ax.set_ylabel("TVD para o baseline livre correspondente")
    ax.set_title("Mudança da distribuição: exata versus observada")
    ax.legend()
    fig.tight_layout()
    fig.savefig(figure_dir / "theta17_distribution_tvd_exact_vs_shots.png", dpi=180)
    plt.show()

    fig, ax = plt.subplots(figsize=(9, 5))
    for condition, label in [
        ("fixed_optimal_interval", "intervalo ótimo"),
        ("fixed_shifted_control", "controle deslocado T/2"),
    ]:
        subset = fixed_summary.loc[
            fixed_summary["condition"].eq(condition)
        ].sort_values("grid_index")
        ax.plot(
            subset["grid_index"],
            subset["tvd_shots_vs_exact_median"],
            marker="o",
            label=label,
        )
    ax.set_xlabel("posição na grade angular")
    ax.set_ylabel("TVD entre 4096 shots e Statevector")
    ax.set_title("Erro amostral da distribuição final")
    ax.legend()
    fig.tight_layout()
    fig.savefig(figure_dir / "theta17_shots_vs_exact_tvd.png", dpi=180)
    plt.show()
else:
    print(
        "Experimento theta_17 não executado. Defina "
        "RUN_THETA17_FIXED_EXPERIMENT=True depois de carregar/completar o banco k=4."
    )


# Parte VII — tabelas para análise quântica e Transformer

O objetivo desta seção é separar entidades que não devem ser confundidas:

1. **problema/Hamiltoniano:** retornos, covariância, $k$, Ising, solução exata e gaps de pares;
2. **parâmetro estrutural:** porta, qubits, distância, ocorrências e período;
3. **execução VQE:** ponto inicial, $\theta$ final, energia, $P(x^*)$ e custo;
4. **ligação par–$\theta$:** quais parâmetros tocam um ou os dois qubits de cada par.

O Transformer deverá receber grupos completos por `problem_id`. Um split aleatório entre vetores do mesmo Hamiltoniano produziria vazamento.


In [ ]:
# ============================================================
# 14. CONSTRUIR BANCOS NORMALIZADOS
# ============================================================

problem_rows = []
pair_frames = []
parameter_frames = []
theta_frames = []
link_frames = []

for k_value, context in problem_contexts.items():
    ising = context["ising"]
    coeffs = [float(np.real(value)) for value in np.asarray(ising.coeffs)]
    labels = [str(value) for value in ising.paulis.to_labels()]
    classical = context["classical"]

    problem_rows.append({
        "problem_id": context["problem_id"],
        "hamiltonian_id": context["hamiltonian_id"],
        "n": context["n"],
        "k": context["k"],
        "q_value": Q_VALUE,
        "risk_free": RISK_FREE,
        "tickers": tuple(tickers),
        "returns": tuple(float(v) for v in mu),
        "covariance_flat": tuple(float(v) for v in sigma.reshape(-1)),
        "ising_pauli_labels": tuple(labels),
        "ising_coefficients": tuple(coeffs),
        "ising_offset": float(context["offset"]),
        "exact_energy": float(context["exact_energy"]),
        "exact_bitstrings": tuple(context["exact_bitstrings"]),
        "selected_assets": tuple(classical["selected_assets"]),
        "n_parameters": int(context["n_parameters"]),
        "split_group": context["problem_id"],
    })

    local_pairs = classical["pair_table"].copy()
    local_pairs.insert(0, "problem_id", context["problem_id"])
    local_pairs.insert(1, "hamiltonian_id", context["hamiltonian_id"])
    local_pairs.insert(2, "n", context["n"])
    local_pairs.insert(3, "k", context["k"])
    pair_frames.append(local_pairs)

    local_parameters = context["parameter_map"].copy()
    local_parameters.insert(0, "problem_id", context["problem_id"])
    local_parameters.insert(1, "hamiltonian_id", context["hamiltonian_id"])
    parameter_frames.append(local_parameters)

    bank_df = bank_frames[k_value]
    if not bank_df.empty:
        local_theta = bank_df.copy()
        local_theta["theta_sin"] = local_theta["best_parameters"].map(
            lambda theta: tuple(float(v) for v in np.sin(np.asarray(theta, dtype=float)))
        )
        local_theta["theta_cos"] = local_theta["best_parameters"].map(
            lambda theta: tuple(float(v) for v in np.cos(np.asarray(theta, dtype=float)))
        )
        local_theta["split_group"] = local_theta["problem_id"]
        theta_frames.append(local_theta)

    for _, pair_row in local_pairs.iterrows():
        pair_qubits = {int(pair_row["asset_index_i"]), int(pair_row["asset_index_j"])}
        for _, parameter_row in local_parameters.iterrows():
            physical_qubits = set(int(v) for v in parameter_row["physical_qubits"])
            intersection = pair_qubits.intersection(physical_qubits)
            if not intersection:
                continue
            link_frames.append(pd.DataFrame([{
                "problem_id": context["problem_id"],
                "hamiltonian_id": context["hamiltonian_id"],
                "n": context["n"],
                "k": context["k"],
                "pair": pair_row["pair"],
                "asset_index_i": int(pair_row["asset_index_i"]),
                "asset_index_j": int(pair_row["asset_index_j"]),
                "G_ij": float(pair_row["G_ij"]),
                "G_ij_relative": float(pair_row["G_ij_relative"]),
                "theta_index": int(parameter_row["theta_index"]),
                "ansatz_gate_type": parameter_row["ansatz_gate_type"],
                "physical_qubits": tuple(parameter_row["physical_qubits"]),
                "touches_any_pair_qubit": True,
                "touches_both_pair_qubits": bool(pair_qubits.issubset(physical_qubits)),
                "n_pair_qubits_touched": int(len(intersection)),
                "angular_period": float(parameter_row["angular_period"]),
                "periodicity_structural_class": parameter_row["periodicity_structural_class"],
            }]))

problem_bank_df = pd.DataFrame(problem_rows).sort_values("k").reset_index(drop=True)
pair_bank_df = pd.concat(pair_frames, ignore_index=True)
parameter_bank_df = pd.concat(parameter_frames, ignore_index=True)
theta_bank_df = pd.concat(theta_frames, ignore_index=True) if theta_frames else pd.DataFrame()
pair_theta_link_df = pd.concat(link_frames, ignore_index=True) if link_frames else pd.DataFrame()

print("problemas:", len(problem_bank_df))
print("pares:", len(pair_bank_df))
print("parâmetros estruturais:", len(parameter_bank_df))
print("execuções theta disponíveis:", len(theta_bank_df))
print("ligações par-theta:", len(pair_theta_link_df))
display(problem_bank_df[["problem_id", "k", "exact_energy", "n_parameters", "selected_assets"]])
display(pair_theta_link_df.head(20))


## 15. Leitura correta para o Transformer

A versão 19.5 produz o **esquema** de treinamento, mas não declara generalização com apenas uma janela financeira.

### Entradas recomendadas

- coeficientes do Hamiltoniano ou, de forma equivalente, retorno e covariância;
- cardinalidade $k$;
- mapa estrutural do ansatz;
- $G_{ij}$ e penalidades direcionais como features clássicas auxiliares.

### Alvos recomendados

- energia e gap exatos;
- $P(x^*)$;
- bitstring ótimo ou distribuição de bitstrings;
- $\sin\theta$ e $\cos\theta$, nunca MSE cru como único alvo.

### Separação dos dados

Todos os vetores do mesmo `problem_id` pertencem ao mesmo grupo. Para avaliar generalização real serão necessários múltiplos Hamiltonianos, obtidos por diferentes janelas temporais e valores de $q$.


In [ ]:
# ============================================================
# 16. SALVAR TABELAS, AUDITORIAS E MANIFESTO
# ============================================================


def csv_safe(df):
    out = df.copy()
    for column in out.columns:
        if out.empty:
            break
        if out[column].map(lambda value: isinstance(value, (list, tuple, dict, np.ndarray))).any():
            out[column] = out[column].map(
                lambda value: json.dumps(
                    value.tolist() if isinstance(value, np.ndarray) else value,
                    ensure_ascii=False,
                ) if isinstance(value, (list, tuple, dict, np.ndarray)) else value
            )
    return out

# Caso clássico original
csv_safe(enumeration_df).to_csv(classical_dir / "enumeration_k4.csv", index=False)
csv_safe(asset_decision_df).to_csv(classical_dir / "asset_decision_margins_k4.csv", index=False)
csv_safe(pair_table).to_csv(classical_dir / "pair_conditional_energies_k4.csv", index=False)
csv_safe(pair_category_selection_df).to_csv(classical_dir / "pair_category_selection_k4.csv", index=False)
ablation_summary_df.to_csv(classical_dir / "old_margin_ablation_k4.csv", index=False)
(classical_dir / "classical_audit_k4.json").write_text(
    json.dumps(audit, indent=2, ensure_ascii=False), encoding="utf-8"
)

# Tabelas multi-k e quânticas
csv_safe(problem_bank_df).to_csv(transformer_dir / "problem_hamiltonian_bank.csv", index=False)
csv_safe(pair_bank_df).to_csv(transformer_dir / "pair_gap_bank.csv", index=False)
csv_safe(parameter_bank_df).to_csv(transformer_dir / "parameter_structure_bank.csv", index=False)
csv_safe(pair_theta_link_df).to_csv(transformer_dir / "pair_theta_link_bank.csv", index=False)
if not theta_bank_df.empty:
    theta_bank_df.to_pickle(transformer_dir / "theta_execution_bank.pkl")
    csv_safe(theta_bank_df.drop(columns=["counts"], errors="ignore")).to_csv(
        transformer_dir / "theta_execution_bank.csv", index=False
    )

if not theta17_runs_df.empty:
    theta17_runs_df.to_pickle(theta17_dir / "theta17_fixed_execution_bank.pkl")
    csv_safe(
        theta17_runs_df.drop(
            columns=[
                "probability_vector",
                "probability_vector_exact",
                "probability_vector_shots",
                "bitstring_labels",
                "counts_shots",
            ],
            errors="ignore",
        )
    ).to_csv(theta17_dir / "theta17_fixed_execution_bank.csv", index=False)
    csv_safe(theta17_summary_df).to_csv(
        theta17_dir / "theta17_fixed_summary.csv", index=False
    )
    csv_safe(theta17_paired_tests_df).to_csv(
        theta17_dir / "theta17_paired_tests.csv", index=False
    )
    csv_safe(theta17_probability_long_df).to_csv(
        theta17_dir / "theta17_probability_distribution.csv", index=False
    )
    csv_safe(theta17_probability_long_df).to_csv(
        theta17_dir / "theta17_probability_distribution_exact_vs_4096shots.csv",
        index=False,
    )

(theta17_dir / "theta17_interval_metadata.json").write_text(
    json.dumps(theta17_interval_metadata, indent=2, ensure_ascii=False),
    encoding="utf-8",
)

for name, dataframe in {
    "problem_hamiltonian_bank": problem_bank_df,
    "pair_gap_bank": pair_bank_df,
    "parameter_structure_bank": parameter_bank_df,
    "pair_theta_link_bank": pair_theta_link_df,
}.items():
    try:
        dataframe.to_parquet(transformer_dir / f"{name}.parquet", index=False)
    except Exception as exc:
        warnings.warn(f"{name}.parquet não foi salvo: {type(exc).__name__}: {exc}")

n_complete_problems = int(bank_status_df["complete"].sum())
readiness = {
    "schema_ready": True,
    "theta_banks_complete": bool(bank_status_df["complete"].all()),
    "n_problem_ids": int(len(problem_bank_df)),
    "n_distinct_hamiltonians": int(problem_bank_df["hamiltonian_id"].nunique()),
    "architecture_experiments_ready": bool(n_complete_problems > 0),
    "theta17_fixed_experiment_enabled": bool(RUN_THETA17_FIXED_EXPERIMENT),
    "theta17_fixed_experiment_executed": bool(
        theta17_interval_metadata.get("executed", False)
    ),
    "theta17_fixed_experiment_rows": int(len(theta17_runs_df)),
    "theta17_distribution_shots": int(THETA17_DISTRIBUTION_SHOTS),
    "theta17_exact_and_sampled_distributions": bool(
        not theta17_runs_df.empty
        and {"probability_vector_exact", "probability_vector_shots"}.issubset(
            theta17_runs_df.columns
        )
    ),
    "generalization_claim_ready": bool(problem_bank_df["hamiltonian_id"].nunique() >= 8),
    "required_next_step": (
        "gerar múltiplas janelas temporais/q antes de alegar generalização"
        if problem_bank_df["hamiltonian_id"].nunique() < 8
        else "executar split por Hamiltoniano"
    ),
}

manifest = {
    **CONFIG,
    "stock_path": str(stock_path.resolve()),
    "data_hash": DATA_HASH,
    "output_dir": str(output_dir.resolve()),
    "classical_audit": audit,
    "periodicity_audit_k4": periodicity_audit,
    "cardinality_audit": cardinality_audit_df.to_dict("records"),
    "bank_status": bank_status_df.to_dict("records"),
    "theta17_fixed_experiment": theta17_interval_metadata,
    "dataset_readiness": readiness,
}
(output_dir / "run_manifest.json").write_text(
    json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8"
)

print(json.dumps(readiness, indent=2, ensure_ascii=False))
print("arquivos salvos em:", output_dir.resolve())


# Parte VIII — próximos testes, já ligados às saídas desta versão

A versão 19.5.3 termina com a análise clássica, os bancos multi-$k$ e o experimento controlado de $\theta_{17}$. O novo teste não substitui a ponte por Fourier: ele verifica se uma região angular previamente descoberta continua útil quando os demais parâmetros podem se reorganizar.

## Como interpretar o experimento de $\theta_{17}$

A decisão deve usar simultaneamente:

- diferença pareada de $P(x^*)$ exata e observada com 4096 shots entre intervalo ótimo e controle deslocado;
- diferença de gap energético;
- taxa de bitstring ótimo dominante;
- distância de variação total da distribuição completa;
- TVD/JSD entre a distribuição de 4096 shots e o Statevector;
- número de avaliações e tempo do COBYLA.

Uma vantagem apenas em $P(x^*)$, acompanhada de piora forte na energia ou no custo, não deve ser tratada como vitória operacional.

## 19.6 — ponte quântica por Fourier

1. usar `pair_gap_bank.csv` para selecionar pares por $G_{ij}$ e categoria;
2. usar `parameter_structure_bank.csv` para determinar período, porta e qubits;
3. usar o resultado de $\theta_{17}$ como primeiro caso de consistência entre sensibilidade global e utilidade após reotimização;
4. varrer cada $\theta$ no seu domínio periódico;
5. calcular espectro de $\langle H\rangle$ e $P(x^*)$;
6. testar $\rho(G_{ij},F^E)$, $\rho(G_{ij},F^P)$ e $\rho(F^P,S_{causal})$;
7. executar FFT 2D apenas nos pares de parâmetros pré-selecionados.

## Experimento fatorial de posição

- manter a mesma posição física e variar o par financeiro;
- manter o mesmo par e variar a posição;
- modelar separadamente efeito do par, efeito da posição e interação.

## Teste operacional final

Comparar, com o mesmo orçamento:

- COBYLA normal;
- COBYLA com $\theta$ fixo em região prevista;
- priorização por $G_{ij}$;
- priorização aleatória;
- oracle causal/Fourier.

Somente essa comparação decide se a informação deve entrar na arquitetura do Transformer.


# Guia de execução

1. Execute até o fim da Parte I e confirme a auditoria histórica do caso $k=4$.
2. Execute a codificação Ising e a auditoria de periodicidade do ansatz.
3. Gere ou carregue o banco definitivo com `GENERATE_OR_RESUME_BANKS=True`.
4. Confirme que o banco de $k=4$ possui pelo menos 20 execuções válidas; o desenho definitivo usa 100.
5. Ative `RUN_THETA17_FIXED_EXPERIMENT=True` e execute a Parte VI.
6. O experimento possui um único desenho: 20 restarts, sete pontos no intervalo ótimo, sete pontos no controle deslocado e um baseline livre por restart.
7. Checkpoints são gravados após cada otimização; uma interrupção não exige reiniciar as execuções concluídas.
8. Examine primeiro `theta17_paired_tests.csv` e `theta17_fixed_summary.csv`. Compare os mapas de calor exato, 4096 shots e residual para verificar quais bitstrings receberam ou perderam probabilidade e quanto dessa mudança é apenas flutuação amostral.
9. Confirme que `tvd_shots_vs_exact` é compatível com o orçamento de 4096 shots e que as conclusões não dependem apenas do histograma amostrado.
10. Somente depois execute as células de exportação dos bancos para o Transformer.

A flag do experimento controla apenas o custo de execução. Ela não cria uma versão piloto ou robusta paralela.
